In [1]:

# ================================================================
# FINAL EXPANDED DATASET: CONSOLIDATION + FULL QA
#
# Original train:       320
# New geographic:        39
# Final train:          359
# Original validation:   80
# Original test:        100 (kept separate)
#
# No Earth Engine downloads.
# No model training.
# No test-set tuning.
# No modification of original datasets or checkpoints.
# ================================================================

from pathlib import Path
from datetime import datetime
import json
import shutil
import hashlib

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# ------------------------------------------------
# 1. PATHS
# ------------------------------------------------

PROJECT = Path("/home/jovyan/vault/LULC_Project")

ORIGINAL = PROJECT / "processed_dataset_complete"
EXPANDED = PROJECT / "processed_dataset_geo_expanded"

FINAL = PROJECT / "processed_dataset_final_expanded"

RAW_ORIGINAL = PROJECT / "data" / "tiles"
RAW_NEW = PROJECT / "data" / "geographic_expansion_raw"

NEW_RAW_MANIFEST = RAW_NEW / "new_tile_manifest.csv"

CLASS_NAMES = {
    0: "Tree cover",
    1: "Shrubland / grassland / moss",
    2: "Cropland",
    3: "Built-up",
    4: "Bare land / snow / ice",
    5: "Water / wetland / mangrove"
}

EXPECTED_IMAGE = (12, 2, 256, 256)
EXPECTED_MASK = (256, 256)
EXPECTED_VALIDITY = (12, 2, 256, 256)

ALLOWED_LABELS = [0, 1, 2, 3, 4, 5, 255]

# Never overwrite an existing final dataset.
if FINAL.exists() and any(FINAL.iterdir()):
    raise FileExistsError(
        f"Final output already exists: {FINAL}\n"
        "Nothing was overwritten. Choose a new output folder."
    )

assert ORIGINAL.exists(), ORIGINAL
assert EXPANDED.exists(), EXPANDED
assert NEW_RAW_MANIFEST.exists(), NEW_RAW_MANIFEST

print("All input paths found.")

# ------------------------------------------------
# 2. VERIFY NEW RAW TILES
# ------------------------------------------------

new_manifest = pd.read_csv(NEW_RAW_MANIFEST)

if "raw_file" not in new_manifest.columns:
    raise ValueError("New raw manifest has no raw_file column.")

new_manifest = new_manifest[
    new_manifest["raw_file"].notna()
].copy()

new_manifest = new_manifest[
    new_manifest["raw_file"].astype(str).str.len() > 0
].copy()

new_raw_files = []

for _, row in new_manifest.iterrows():

    p = Path(row["raw_file"])

    if not p.exists():
        raise FileNotFoundError(p)

    with np.load(p, allow_pickle=False) as data:

        if "s1" not in data or "label" not in data:
            raise ValueError(f"Missing arrays in {p.name}")

        image = data["s1"]
        mask = data["label"]

        if image.shape != EXPECTED_IMAGE:
            raise ValueError(
                f"{p.name}: incorrect raw image shape {image.shape}"
            )

        if mask.shape != EXPECTED_MASK:
            raise ValueError(
                f"{p.name}: incorrect raw mask shape {mask.shape}"
            )

        if not np.isin(
            np.unique(mask), ALLOWED_LABELS
        ).all():
            raise ValueError(
                f"{p.name}: invalid label values {np.unique(mask)}"
            )

    new_raw_files.append(p)

if len(new_raw_files) != 39:
    raise ValueError(
        f"Expected 39 new raw tiles, found {len(new_raw_files)}"
    )

print("New raw tiles verified:", len(new_raw_files))

# ------------------------------------------------
# 3. VERIFY NORMALIZATION COMPATIBILITY
# ------------------------------------------------

metadata_path = (
    EXPANDED / "expansion_metadata.json"
)

if not metadata_path.exists():
    raise FileNotFoundError(metadata_path)

with open(metadata_path) as f:
    expansion_metadata = json.load(f)

normalization = expansion_metadata.get(
    "original_training_statistics_used"
)

if not normalization:
    raise ValueError(
        "Original training normalization statistics "
        "not found in expansion metadata."
    )

print("\nNormalization used for new tiles:")
print(json.dumps(normalization, indent=2))

# ------------------------------------------------
# 4. CREATE FINAL DIRECTORY STRUCTURE
# ------------------------------------------------

for split in ["train", "val", "test"]:

    for folder in ["images", "masks", "validity"]:

        (FINAL / split / folder).mkdir(
            parents=True,
            exist_ok=True
        )

# ------------------------------------------------
# 5. COPY ORIGINAL PROCESSED DATA
#
# Original train and validation remain unchanged.
# Original test is copied only for later final evaluation.
# ------------------------------------------------

def safe_copy_dataset(source_root, destination_root, splits):

    for split in splits:

        for folder in ["images", "masks", "validity"]:

            source = source_root / split / folder
            destination = destination_root / split / folder

            if not source.exists():
                raise FileNotFoundError(source)

            for file in source.glob("*.npy"):

                target = destination / file.name

                if target.exists():
                    raise FileExistsError(
                        f"Duplicate filename: {target}"
                    )

                shutil.copy2(file, target)


safe_copy_dataset(
    ORIGINAL,
    FINAL,
    ["train", "val", "test"]
)

print("\nOriginal processed dataset copied.")

# ------------------------------------------------
# 6. ADD THE 39 NEW PROCESSED TRAINING TILES
#
# These were normalized using original training-only
# median/IQR statistics.
# ------------------------------------------------

new_image_dir = EXPANDED / "train" / "images"
new_mask_dir = EXPANDED / "train" / "masks"
new_valid_dir = EXPANDED / "train" / "validity"

for raw_path in tqdm(
    new_raw_files,
    desc="Adding new geographic tiles"
):

    tile_id = raw_path.stem

    image_path = new_image_dir / f"{tile_id}.npy"
    mask_path = new_mask_dir / f"{tile_id}.npy"
    validity_path = new_valid_dir / f"{tile_id}.npy"

    for p in [image_path, mask_path, validity_path]:
        if not p.exists():
            raise FileNotFoundError(
                f"Missing processed new tile: {p}"
            )

    targets = [
        FINAL / "train" / "images" / image_path.name,
        FINAL / "train" / "masks" / mask_path.name,
        FINAL / "train" / "validity" / validity_path.name
    ]

    sources = [image_path, mask_path, validity_path]

    for src, dst in zip(sources, targets):

        if dst.exists():
            raise FileExistsError(
                f"Duplicate tile in final dataset: {dst}"
            )

        shutil.copy2(src, dst)

print("New geographic tiles added.")

# ------------------------------------------------
# 7. FULL TILE-BY-TILE QA
# ------------------------------------------------

audit_rows = []
class_counts = {
    split: np.zeros(6, dtype=np.int64)
    for split in ["train", "val", "test"]
}

split_counts = {}

for split in ["train", "val", "test"]:

    image_dir = FINAL / split / "images"
    mask_dir = FINAL / split / "masks"
    validity_dir = FINAL / split / "validity"

    image_files = sorted(image_dir.glob("*.npy"))

    if not image_files:
        raise RuntimeError(f"No images in {split}")

    split_counts[split] = len(image_files)

    image_names = {p.name for p in image_files}
    mask_names = {p.name for p in mask_dir.glob("*.npy")}
    validity_names = {
        p.name for p in validity_dir.glob("*.npy")
    }

    if image_names != mask_names:
        raise AssertionError(
            f"{split}: image/mask filename mismatch"
        )

    if image_names != validity_names:
        raise AssertionError(
            f"{split}: image/validity filename mismatch"
        )

    for image_path in tqdm(
        image_files,
        desc=f"Auditing {split}"
    ):

        mask_path = mask_dir / image_path.name
        validity_path = validity_dir / image_path.name

        image = np.load(image_path, mmap_mode="r")
        mask = np.load(mask_path, mmap_mode="r")
        validity = np.load(validity_path, mmap_mode="r")

        # Shape checks.
        assert image.shape == EXPECTED_IMAGE, (
            image_path.name, image.shape
        )

        assert mask.shape == EXPECTED_MASK, (
            mask_path.name, mask.shape
        )

        assert validity.shape == EXPECTED_VALIDITY, (
            validity_path.name, validity.shape
        )

        # Dtype checks.
        assert image.dtype == np.float32
        assert mask.dtype == np.uint8
        assert validity.dtype == np.uint8

        # Finite normalized inputs.
        if not np.isfinite(image).all():
            raise ValueError(
                f"Non-finite normalized values: {image_path.name}"
            )

        # Label checks.
        unique_labels = np.unique(mask)

        if not np.isin(
            unique_labels, ALLOWED_LABELS
        ).all():
            raise ValueError(
                f"Unexpected labels in {mask_path.name}: "
                f"{unique_labels}"
            )

        # Validity checks.
        if not np.isin(
            np.unique(validity), [0, 1]
        ).all():
            raise ValueError(
                f"Invalid validity values: {validity_path.name}"
            )

        # Invalid observations must be zero-filled.
        invalid_values = np.asarray(image)[
            np.asarray(validity) == 0
        ]

        if invalid_values.size and not np.allclose(
            invalid_values, 0.0, atol=1e-6
        ):
            raise ValueError(
                f"Invalid observations not zero-filled: "
                f"{image_path.name}"
            )

        # Per-class counts.
        tile_counts = np.array([
            np.sum(mask == c)
            for c in range(6)
        ], dtype=np.int64)

        class_counts[split] += tile_counts

        labelled = int(tile_counts.sum())

        total_mask_pixels = mask.size

        audit_rows.append({
            "split": split,
            "tile_id": image_path.stem,
            "labelled_pixels": labelled,
            "ignored_pixels": int(
                np.sum(mask == 255)
            ),
            "validity_percentage": float(
                np.asarray(validity).mean() * 100
            ),
            "tree_pixels": int(tile_counts[0]),
            "shrub_pixels": int(tile_counts[1]),
            "cropland_pixels": int(tile_counts[2]),
            "built_pixels": int(tile_counts[3]),
            "bare_pixels": int(tile_counts[4]),
            "water_pixels": int(tile_counts[5]),
            "bare_percentage": (
                100 * tile_counts[4] / labelled
                if labelled else 0
            )
        })

# ------------------------------------------------
# 8. VERIFY EXACT SPLIT COUNTS
# ------------------------------------------------

expected_counts = {
    "train": 359,
    "val": 80,
    "test": 100
}

print("\nFinal split counts:")

for split, expected in expected_counts.items():

    actual = split_counts[split]

    print(f"{split}: {actual}")

    if actual != expected:
        raise ValueError(
            f"{split}: expected {expected}, got {actual}"
        )

# ------------------------------------------------
# 9. CLASS DISTRIBUTION
# ------------------------------------------------

distribution_rows = []
distribution = {}

for split in ["train", "val", "test"]:

    total = int(class_counts[split].sum())

    distribution[split] = {}

    for c, name in CLASS_NAMES.items():

        count = int(class_counts[split][c])

        percentage = (
            100 * count / total
            if total else 0
        )

        distribution[split][str(c)] = {
            "class_name": name,
            "pixels": count,
            "percentage": percentage
        }

        distribution_rows.append({
            "split": split,
            "class_id": c,
            "class_name": name,
            "pixels": count,
            "percentage": percentage
        })

audit_df = pd.DataFrame(audit_rows)

distribution_df = pd.DataFrame(distribution_rows)

audit_df.to_csv(
    FINAL / "tile_level_audit.csv",
    index=False
)

distribution_df.to_csv(
    FINAL / "class_distribution.csv",
    index=False
)

# ------------------------------------------------
# 10. NEW GEOGRAPHIC TILE AUDIT
# ------------------------------------------------

new_ids = {p.stem for p in new_raw_files}

new_audit = audit_df[
    audit_df["tile_id"].isin(new_ids)
].copy()

if len(new_audit) != 39:
    raise ValueError(
        f"Expected 39 new tiles in final audit, got {len(new_audit)}"
    )

print("\nNew geographic tile class distribution:")

new_bare = new_audit["bare_pixels"].sum()
new_labelled = new_audit["labelled_pixels"].sum()

print(
    "Bare land percentage across new tiles:",
    round(100 * new_bare / new_labelled, 3)
)

print(
    "New tile median bare percentage:",
    round(new_audit["bare_percentage"].median(), 3)
)

print(
    "New tiles with >=10% bare:",
    int((new_audit["bare_percentage"] >= 10).sum())
)

print(
    "New tiles with >=20% bare:",
    int((new_audit["bare_percentage"] >= 20).sum())
)

# ------------------------------------------------
# 11. NORMALIZATION CONSISTENCY CHECK
# ------------------------------------------------

# Check that the 39 new processed arrays are exactly the
# arrays previously generated in the expansion pipeline.
# This catches accidental copying or alteration.

for tile_id in new_ids:

    for folder in ["images", "masks", "validity"]:

        source = EXPANDED / "train" / folder / f"{tile_id}.npy"
        target = FINAL / "train" / folder / f"{tile_id}.npy"

        a = np.load(source)
        b = np.load(target)

        if not np.array_equal(a, b):
            raise ValueError(
                f"Copy mismatch for {tile_id}/{folder}"
            )

print("\nNew tile copy consistency: PASS")

# ------------------------------------------------
# 12. SAVE FINAL METADATA
# ------------------------------------------------

metadata = {
    "dataset_name": "Sentinel-1 LULC Expanded Final Dataset",
    "created_at": datetime.now().isoformat(),
    "path": str(FINAL),
    "splits": split_counts,
    "input_shape": list(EXPECTED_IMAGE),
    "mask_shape": list(EXPECTED_MASK),
    "validity_shape": list(EXPECTED_VALIDITY),
    "class_names": CLASS_NAMES,
    "ignore_label": 255,
    "normalization": {
        "method": "Original training-only median/IQR",
        "statistics": normalization,
        "recomputed": False,
        "reason": (
            "Preserves input normalization compatibility "
            "with the existing Experiment 7 checkpoint."
        )
    },
    "new_geographic_tiles": len(new_ids),
    "test_set_used_for_training": False,
    "test_set_used_for_selection": False,
    "original_dataset_modified": False,
    "all_tile_qa_passed": True,
    "class_distribution": distribution
}

with open(FINAL / "dataset_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

# ------------------------------------------------
# 13. FINAL REPORT
# ------------------------------------------------

print("\n" + "=" * 72)
print("FINAL EXPANDED DATASET QA COMPLETE")
print("=" * 72)

print("\nSplit sizes:")

for split, count in split_counts.items():
    print(f"{split.upper():5s}: {count}")

print("\nClass distribution:")

for split in ["train", "val", "test"]:

    print(f"\n{split.upper()}")

    for c, name in CLASS_NAMES.items():

        pct = distribution[split][str(c)]["percentage"]

        print(f"{name:36s} {pct:8.3f}%")

print("\nNew geographic tiles:", len(new_ids))
print("All images, masks and validity arrays passed QA.")
print("Original datasets untouched.")
print("Test set not used for training or selection.")

print("\nFINAL DATASET:")
print(FINAL)


All input paths found.
New raw tiles verified: 39

Normalization used for new tiles:
{
  "VV": {
    "median": -10.296875,
    "q25": -13.8203125,
    "q75": -8.0703125,
    "iqr": 5.75
  },
  "VH": {
    "median": -16.9375,
    "q25": -21.4375,
    "q75": -14.671875,
    "iqr": 6.765625
  }
}

Original processed dataset copied.


Adding new geographic tiles:   0%|          | 0/39 [00:00<?, ?it/s]

New geographic tiles added.


Auditing train:   0%|          | 0/359 [00:00<?, ?it/s]

Auditing val:   0%|          | 0/80 [00:00<?, ?it/s]

Auditing test:   0%|          | 0/100 [00:00<?, ?it/s]


Final split counts:
train: 359
val: 80
test: 100

New geographic tile class distribution:
Bare land percentage across new tiles: 74.356
New tile median bare percentage: 93.936
New tiles with >=10% bare: 39
New tiles with >=20% bare: 36

New tile copy consistency: PASS

FINAL EXPANDED DATASET QA COMPLETE

Split sizes:
TRAIN: 359
VAL  : 80
TEST : 100

Class distribution:

TRAIN
Tree cover                             24.322%
Shrubland / grassland / moss           18.653%
Cropland                               13.378%
Built-up                               16.133%
Bare land / snow / ice                 10.043%
Water / wetland / mangrove             17.470%

VAL
Tree cover                             26.376%
Shrubland / grassland / moss           19.182%
Cropland                               20.421%
Built-up                               13.861%
Bare land / snow / ice                  1.455%
Water / wetland / mangrove             18.705%

TEST
Tree cover                             26.597

In [3]:

# =============================================================================
# EXPERIMENT 20 — ADD REAL GEOGRAPHIC TRAINING TILES
# Sentinel-1 monthly composites + ESA WorldCover 2021 labels
#
# Uses:
#   /home/jovyan/vault/LULC_Project/data/metadata/sites.csv
#   /home/jovyan/vault/LULC_Project/data/metadata/tile_manifest_candidates.csv
#
# Exports:
#   12 monthly Sentinel-1 VV/VH composites + 1 WorldCover label band
#   256 x 256 pixels, 30m, UTM projection
#
# IMPORTANT:
#   - Only TRAIN cities are used.
#   - Existing validation and test data are untouched.
#   - New tiles are real geographic samples, not synthetic samples.
# =============================================================================


# =============================================================================
# 1. IMPORTS
# =============================================================================

import os
import math
import random
import time
import json
import warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd

import ee

warnings.filterwarnings("ignore", category=FutureWarning)

# =============================================================================
# 2. CONFIGURATION
# =============================================================================

PROJECT_DIR = Path("/home/jovyan/vault/LULC_Project")

METADATA_DIR = PROJECT_DIR / "data" / "metadata"

SITES_CSV = METADATA_DIR / "sites.csv"
TILE_MANIFEST_CSV = METADATA_DIR / "tile_manifest_candidates.csv"

OUTPUT_DIR = PROJECT_DIR / "exp20_new_real_tiles"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Earth Engine project
EE_PROJECT = "final-project-513714"

# Original tile geometry
TILE_PIXELS = 256
RESOLUTION_M = 30
TILE_SIZE_M = TILE_PIXELS * RESOLUTION_M

HALF_TILE_M = TILE_SIZE_M / 2

# Candidate generation
RANDOM_SEED = 42

CANDIDATES_PER_CITY = 30

MAX_NEW_TILES = 120

MAX_TILES_PER_CITY = 5

CITY_SEARCH_RADIUS_KM = 40

# Conservative separation from existing candidate tile centres.
# A 7.68 km square tile has a diagonal of approximately 10.86 km.
MIN_EXISTING_DISTANCE_KM = 11.0

# Minimum separation between newly generated candidate centres.
MIN_NEW_DISTANCE_KM = 11.0

# SAR period
START_DATE = "2021-01-01"
END_DATE = "2022-01-01"

# Drive export
DRIVE_FOLDER = "LULC_Project_Exp20_New_Tiles"

# Number of Earth Engine export tasks to start in one batch.
# Keep this moderate to avoid overwhelming task submission.
EXPORT_BATCH_SIZE = 20

# Six merged classes
CLASS_NAMES = [
    "Tree",
    "Shrub_Grass_Moss",
    "Cropland",
    "Built_up",
    "Bare_Snow_Ice",
    "Water_Wetland_Mangrove"
]

CLASS_IDS = [0, 1, 2, 3, 4, 5]

# ESA WorldCover mapping
WORLD_COVER_CODES = [
    10, 20, 30, 40, 50, 60, 70, 80, 90, 95, 100
]

MERGED_CLASS_CODES = [
    0, 1, 1, 2, 3, 4, 4, 5, 5, 5, 1
]

# Starting approximate class proportions in the current expanded
# training dataset (359 tiles).
#
# These are used only to prioritize additional samples.
# They are not used as validation/test information.

CURRENT_TRAIN_PERCENTAGES = np.array([
    24.322,
    18.653,
    13.378,
    16.133,
    10.043,
    17.470
], dtype=np.float64)

assert abs(CURRENT_TRAIN_PERCENTAGES.sum() - 100) < 0.1

# Target is equal representation as a collection objective.
# The final real-world pixel distribution must still be reported honestly.
TARGET_PERCENTAGES = np.ones(6, dtype=np.float64) * (100 / 6)

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print("=" * 85)
print("EXPERIMENT 20 — REAL GEOGRAPHIC DATA COLLECTION")
print("=" * 85)

print("Project:", PROJECT_DIR)
print("Metadata:", METADATA_DIR)
print("Output:", OUTPUT_DIR)
print("Tile size:", TILE_PIXELS, "x", TILE_PIXELS)
print("Resolution:", RESOLUTION_M, "m")
print("Tile width:", TILE_SIZE_M / 1000, "km")


# =============================================================================
# 3. INITIALIZE EARTH ENGINE
# =============================================================================

try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)

print("\nEarth Engine initialized:", EE_PROJECT)


# =============================================================================
# 4. LOAD THE ACTUAL PROJECT METADATA
# =============================================================================

if not SITES_CSV.exists():
    raise FileNotFoundError(f"Missing file: {SITES_CSV}")

if not TILE_MANIFEST_CSV.exists():
    raise FileNotFoundError(f"Missing file: {TILE_MANIFEST_CSV}")

sites = pd.read_csv(SITES_CSV)

tile_manifest = pd.read_csv(TILE_MANIFEST_CSV)

# Support either city or site column naming.
if "city" not in sites.columns and "site" in sites.columns:
    sites = sites.rename(columns={"site": "city"})

if "city" not in tile_manifest.columns and "site" in tile_manifest.columns:
    tile_manifest = tile_manifest.rename(columns={"site": "city"})

required_sites = {
    "city", "latitude", "longitude", "split"
}

required_tiles = {
    "city", "latitude", "longitude", "split"
}

missing_sites = required_sites - set(sites.columns)
missing_tiles = required_tiles - set(tile_manifest.columns)

if missing_sites:
    raise ValueError(
        f"sites.csv is missing columns: {missing_sites}\n"
        f"Available: {list(sites.columns)}"
    )

if missing_tiles:
    raise ValueError(
        f"tile_manifest_candidates.csv is missing columns: {missing_tiles}\n"
        f"Available: {list(tile_manifest.columns)}"
    )

sites["split"] = sites["split"].astype(str).str.lower()
tile_manifest["split"] = tile_manifest["split"].astype(str).str.lower()

train_cities = sites[
    sites["split"] == "train"
].copy().reset_index(drop=True)

if len(train_cities) != 32:
    raise ValueError(
        f"Expected 32 training cities, found {len(train_cities)}. "
        "Check the original sites.csv."
    )

# Use all original planned candidate centres as conservative exclusion zones.
existing_aois = tile_manifest[
    ["latitude", "longitude", "split"]
].copy()

existing_aois["latitude"] = pd.to_numeric(
    existing_aois["latitude"], errors="coerce"
)

existing_aois["longitude"] = pd.to_numeric(
    existing_aois["longitude"], errors="coerce"
)

existing_aois = existing_aois.dropna(
    subset=["latitude", "longitude"]
)

existing_coordinates = list(
    zip(
        existing_aois["latitude"].astype(float),
        existing_aois["longitude"].astype(float)
    )
)

print("\nMetadata loaded successfully.")
print("Training cities:", len(train_cities))
print("Original candidate centres:", len(existing_coordinates))
print("\nTraining cities:")
print(train_cities["city"].tolist())


# =============================================================================
# 5. GEOGRAPHIC HELPERS
# =============================================================================

EARTH_RADIUS_KM = 6371.0088


def haversine_km(lat1, lon1, lat2, lon2):
    """Distance between two geographic coordinates in kilometres."""

    lat1 = math.radians(float(lat1))
    lon1 = math.radians(float(lon1))

    lat2 = math.radians(float(lat2))
    lon2 = math.radians(float(lon2))

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        math.sin(dlat / 2) ** 2
        + math.cos(lat1)
        * math.cos(lat2)
        * math.sin(dlon / 2) ** 2
    )

    return 2 * EARTH_RADIUS_KM * math.asin(
        min(1.0, math.sqrt(a))
    )


def too_close_to_existing(lat, lon):
    """Reject points close to any original candidate centre."""

    for old_lat, old_lon in existing_coordinates:
        distance = haversine_km(
            lat, lon, old_lat, old_lon
        )

        if distance < MIN_EXISTING_DISTANCE_KM:
            return True

    return False


def too_close_to_new(lat, lon, selected_points):
    """Reject points close to newly generated candidates."""

    for old_lat, old_lon in selected_points:
        distance = haversine_km(
            lat, lon, old_lat, old_lon
        )

        if distance < MIN_NEW_DISTANCE_KM:
            return True

    return False


def utm_epsg_for_location(lat, lon):
    """Calculate the UTM EPSG code for a geographic location."""

    zone = int((float(lon) + 180) / 6) + 1
    zone = max(1, min(zone, 60))

    if float(lat) >= 0:
        return f"EPSG:{32600 + zone}"

    return f"EPSG:{32700 + zone}"


def random_location_near_city(lat, lon, radius_km):
    """
    Generate a random location within an approximate circular radius.
    This is candidate generation only; WorldCover screening follows.
    """

    angle = random.uniform(0, 2 * math.pi)

    # sqrt gives approximately uniform sampling over the circle.
    distance = radius_km * math.sqrt(random.random())

    north_km = distance * math.cos(angle)
    east_km = distance * math.sin(angle)

    new_lat = float(lat) + north_km / 111.32

    cos_lat = max(
        0.01,
        abs(math.cos(math.radians(float(lat))))
    )

    new_lon = float(lon) + east_km / (
        111.32 * cos_lat
    )

    return new_lat, new_lon


# =============================================================================
# 6. GENERATE NEW CANDIDATE CENTRES
# =============================================================================

print("\nGenerating candidate locations...")

candidate_rows = []

new_candidate_points = []

for _, city_row in train_cities.iterrows():

    city = str(city_row["city"])

    city_lat = float(city_row["latitude"])
    city_lon = float(city_row["longitude"])

    epsg = utm_epsg_for_location(
        city_lat, city_lon
    )

    accepted = 0
    attempts = 0

    max_attempts = CANDIDATES_PER_CITY * 100

    while (
        accepted < CANDIDATES_PER_CITY
        and attempts < max_attempts
    ):

        attempts += 1

        lat, lon = random_location_near_city(
            city_lat,
            city_lon,
            CITY_SEARCH_RADIUS_KM
        )

        if not (-80 <= lat <= 84):
            continue

        if too_close_to_existing(lat, lon):
            continue

        if too_close_to_new(
            lat, lon, new_candidate_points
        ):
            continue

        new_candidate_points.append((lat, lon))

        candidate_rows.append({
            "candidate_id": f"E20_{city}_{accepted + 1:03d}",
            "city": city,
            "latitude": lat,
            "longitude": lon,
            "utm_epsg": epsg
        })

        accepted += 1

    print(
        f"{city:20s} "
        f"{accepted:3d}/{CANDIDATES_PER_CITY} candidates"
    )

candidates = pd.DataFrame(candidate_rows)

if candidates.empty:
    raise RuntimeError(
        "No candidate locations were generated. "
        "Check the metadata and geographic exclusion settings."
    )

candidate_csv = OUTPUT_DIR / "exp20_candidate_locations.csv"

candidates.to_csv(candidate_csv, index=False)

print("\nGenerated candidates:", len(candidates))
print("Saved:", candidate_csv)


# =============================================================================
# 7. LOAD WORLD COVER AND PREPARE CLASS LABELS
# =============================================================================

worldcover = ee.ImageCollection(
    "ESA/WorldCover/v200"
).first().select("Map")

worldcover_merged = worldcover.remap(
    WORLD_COVER_CODES,
    MERGED_CLASS_CODES,
    255
).rename("label").toUint8()

print("\nWorldCover 2021 loaded.")


# =============================================================================
# 8. SCREEN CANDIDATES USING REAL WORLD COVER
# =============================================================================

def screen_city_candidates(city_df):
    """
    Screen all candidate tiles for one city using WorldCover.

    Each candidate is measured on its own UTM 30m grid.
    """

    city = str(city_df.iloc[0]["city"])

    epsg = str(city_df.iloc[0]["utm_epsg"])

    projection = ee.Projection(epsg)

    features = []

    for _, row in city_df.iterrows():

        point = ee.Geometry.Point([
            float(row["longitude"]),
            float(row["latitude"])
        ])

        feature = ee.Feature(
            point,
            {
                "candidate_id": str(row["candidate_id"]),
                "city": city,
                "latitude": float(row["latitude"]),
                "longitude": float(row["longitude"]),
                "utm_epsg": epsg
            }
        )

        features.append(feature)

    feature_collection = ee.FeatureCollection(features)

    def add_class_histogram(feature):

        geom = feature.geometry().transform(
            projection, 1
        )

        rectangle = geom.buffer(
            0
        ).bounds(
            1,
            projection
        )

        # Construct exact square using projected centre coordinates.
        centre = geom.coordinates()

        x = ee.Number(centre.get(0))
        y = ee.Number(centre.get(1))

        xmin = x.subtract(HALF_TILE_M)
        xmax = x.add(HALF_TILE_M)

        ymin = y.subtract(HALF_TILE_M)
        ymax = y.add(HALF_TILE_M)

        tile_region = ee.Geometry.Rectangle(
            [xmin, ymin, xmax, ymax],
            projection,
            False
        )

        histogram = worldcover_merged.reduceRegion(
            reducer=ee.Reducer.frequencyHistogram(),
            geometry=tile_region,
            crs=projection,
            scale=RESOLUTION_M,
            maxPixels=100000,
            tileScale=2
        )

        return feature.set({
            "class_histogram": histogram.get("label")
        })

    screened = feature_collection.map(
        add_class_histogram
    )

    result = screened.getInfo()

    output_rows = []

    for feature in result["features"]:

        props = feature["properties"]

        hist = props.get("class_histogram")

        if not isinstance(hist, dict):
            continue

        counts = np.zeros(6, dtype=np.float64)

        for class_id in CLASS_IDS:

            counts[class_id] = float(
                hist.get(str(class_id), 0)
            )

        total = counts.sum()

        if total <= 0:
            continue

        percentages = counts / total * 100

        row = {
            "candidate_id": props["candidate_id"],
            "city": props["city"],
            "latitude": props["latitude"],
            "longitude": props["longitude"],
            "utm_epsg": props["utm_epsg"],
            "valid_pixels": int(total)
        }

        for class_id, class_name in enumerate(CLASS_NAMES):

            row[f"class_{class_id}_pixels"] = int(
                counts[class_id]
            )

            row[f"class_{class_id}_pct"] = float(
                percentages[class_id]
            )

        output_rows.append(row)

    return output_rows


print("\nScreening candidates with ESA WorldCover...")

screened_rows = []

for city, city_df in candidates.groupby("city"):

    print(
        f"\nScreening {city}: {len(city_df)} candidates"
    )

    try:

        rows = screen_city_candidates(
            city_df.reset_index(drop=True)
        )

        screened_rows.extend(rows)

        print(
            f"  Successfully screened: {len(rows)}"
        )

    except Exception as error:

        print(
            f"  ERROR screening {city}: {error}"
        )

screened_df = pd.DataFrame(screened_rows)

if screened_df.empty:
    raise RuntimeError(
        "WorldCover screening returned no usable candidates."
    )

screened_csv = OUTPUT_DIR / "exp20_screened_candidates.csv"

screened_df.to_csv(screened_csv, index=False)

print("\nSuccessfully screened:", len(screened_df))
print("Saved:", screened_csv)


# =============================================================================
# 9. SELECT TILES TO IMPROVE CLASS REPRESENTATION
# =============================================================================

def class_percentages_from_counts(counts):
    total = float(np.sum(counts))

    if total <= 0:
        return np.zeros(6)

    return counts / total * 100


def balance_objective(percentages):
    """
    Lower is better.

    Measures deviation from equal class representation.
    This is a selection heuristic, not a claim that the final dataset
    should represent the real world's land-cover distribution.
    """

    return float(
        np.mean(
            (percentages - TARGET_PERCENTAGES) ** 2
        )
    )


# Approximate existing pixel counts from the current expanded dataset.
# 359 tiles x 256 x 256 pixels.
# The proportions are the measured expanded-training proportions.

CURRENT_TRAIN_TILES = 359

APPROX_EXISTING_PIXELS = (
    CURRENT_TRAIN_TILES * TILE_PIXELS * TILE_PIXELS
)

current_counts = (
    CURRENT_TRAIN_PERCENTAGES / 100
) * APPROX_EXISTING_PIXELS

working_counts = current_counts.copy()

selected_rows = []

city_selection_counts = defaultdict(int)

remaining = screened_df.copy().reset_index(drop=True)

print("\nStarting class distribution:")
for i, name in enumerate(CLASS_NAMES):
    print(
        f"{name:25s}: "
        f"{CURRENT_TRAIN_PERCENTAGES[i]:6.2f}%"
    )

initial_objective = balance_objective(
    class_percentages_from_counts(working_counts)
)

print(
    "\nInitial balance objective:",
    round(initial_objective, 4)
)

for selection_number in range(MAX_NEW_TILES):

    best_index = None
    best_score = None
    best_new_counts = None

    current_objective = balance_objective(
        class_percentages_from_counts(working_counts)
    )

    for idx, row in remaining.iterrows():

        city = str(row["city"])

        if city_selection_counts[city] >= MAX_TILES_PER_CITY:
            continue

        tile_counts = np.array([
            row[f"class_{i}_pixels"]
            for i in range(6)
        ], dtype=np.float64)

        if tile_counts.sum() <= 0:
            continue

        proposed_counts = working_counts + tile_counts

        proposed_percentages = (
            class_percentages_from_counts(
                proposed_counts
            )
        )

        score = balance_objective(
            proposed_percentages
        )

        # Select only candidates that improve the objective.
        if score >= current_objective:
            continue

        if best_score is None or score < best_score:

            best_score = score
            best_index = idx
            best_new_counts = proposed_counts

    if best_index is None:

        print(
            "\nNo further candidate improves the balance objective."
        )

        break

    selected = remaining.loc[best_index].copy()

    selected_rows.append(selected)

    working_counts = best_new_counts

    city_selection_counts[str(selected["city"])] += 1

    remaining = remaining.drop(
        index=best_index
    ).reset_index(drop=True)

    current_percentages = class_percentages_from_counts(
        working_counts
    )

    print(
        f"\nSelected {len(selected_rows):3d}: "
        f"{selected['candidate_id']} | "
        f"{selected['city']}"
    )

    print(
        "Class distribution:",
        np.round(current_percentages, 2)
    )

selected_df = pd.DataFrame(selected_rows)

if selected_df.empty:
    raise RuntimeError(
        "The selection algorithm found no candidates that improve "
        "the current class-balance objective. Inspect the screened "
        "candidate distributions before changing selection settings."
    )

selected_csv = OUTPUT_DIR / "exp20_selected_tiles.csv"

selected_df.to_csv(selected_csv, index=False)

print("\n" + "=" * 85)
print("SELECTION COMPLETE")
print("=" * 85)

print("Selected tiles:", len(selected_df))
print("Selected cities:", selected_df["city"].nunique())

print("\nTiles per city:")
print(selected_df["city"].value_counts())

final_percentages = class_percentages_from_counts(
    working_counts
)

print("\nEstimated class distribution after selection:")

for i, name in enumerate(CLASS_NAMES):

    print(
        f"{name:25s}: "
        f"{final_percentages[i]:6.2f}%"
    )

print("\nSaved:", selected_csv)


# =============================================================================
# 10. CREATE MONTHLY SENTINEL-1 COMPOSITES
# =============================================================================

s1 = (
    ee.ImageCollection("COPERNICUS/S1_GRD")
    .filterDate(START_DATE, END_DATE)
    .filter(ee.Filter.eq("instrumentMode", "IW"))
    .filter(ee.Filter.listContains(
        "transmitterReceiverPolarisation", "VV"
    ))
    .filter(ee.Filter.listContains(
        "transmitterReceiverPolarisation", "VH"
    ))
    .select(["VV", "VH"])
)

print("\nSentinel-1 collection prepared.")


def make_monthly_composite(region, year, month):
    """
    Create monthly median VV/VH composite.
    Sentinel-1 GRD VV/VH values are already in dB.
    """

    start = ee.Date.fromYMD(
        year, month, 1
    )

    end = start.advance(
        1, "month"
    )

    collection = (
        s1.filterDate(start, end)
        .filterBounds(region)
    )

    empty_image = (
        ee.Image.constant([-9999, -9999])
        .rename(["VV", "VH"])
        .toFloat()
    )

    composite = ee.Image(
        ee.Algorithms.If(
            collection.size().gt(0),
            collection.median().select(["VV", "VH"]),
            empty_image
        )
    )

    return composite.rename([
        f"VV_{month:02d}",
        f"VH_{month:02d}"
    ])


# =============================================================================
# 11. BUILD THE 25-BAND EXPORT IMAGE
# =============================================================================

def build_export_image(row):
    """
    Construct:
      VV_01, VH_01, ..., VV_12, VH_12, label

    The image is exported in the candidate's UTM projection.
    """

    lat = float(row["latitude"])
    lon = float(row["longitude"])

    epsg = str(row["utm_epsg"])

    projection = ee.Projection(epsg)

    centre = ee.Geometry.Point(
        [lon, lat]
    ).transform(
        projection, 1
    )

    xy = centre.coordinates()

    x = ee.Number(xy.get(0))
    y = ee.Number(xy.get(1))

    xmin = x.subtract(HALF_TILE_M)
    xmax = x.add(HALF_TILE_M)

    ymin = y.subtract(HALF_TILE_M)
    ymax = y.add(HALF_TILE_M)

    region = ee.Geometry.Rectangle(
        [xmin, ymin, xmax, ymax],
        projection,
        False
    )

    monthly_bands = []

    for month in range(1, 13):

        composite = make_monthly_composite(
            region,
            2021,
            month
        )

        monthly_bands.append(
            composite.clip(region)
        )

    sar_stack = monthly_bands[0]

    for image in monthly_bands[1:]:

        sar_stack = sar_stack.addBands(
            image
        )

    # WorldCover label: default nearest-neighbour resampling.
    label = (
        worldcover_merged
        .clip(region)
        .rename("label")
    )

    # Cast all bands to float32 for a consistent GeoTIFF.
    # SAR missing observations remain -9999.
    # Labels remain integer-valued and can be converted to uint8
    # during preprocessing.
    export_image = (
        sar_stack
        .addBands(label)
        .toFloat()
        .clip(region)
    )

    # Exact 256 x 256 grid at 30 metres.
    crs_transform = [
        RESOLUTION_M, 0, xmin,
        0, -RESOLUTION_M, ymax
    ]

    return export_image, region, epsg, crs_transform


# =============================================================================
# 12. SUBMIT EXPORT TASKS TO GOOGLE DRIVE
# =============================================================================

export_records = []

print("\nPreparing Earth Engine export tasks...")

for index, row in selected_df.reset_index(drop=True).iterrows():

    candidate_id = str(row["candidate_id"])

    city = str(row["city"])

    print(
        f"\nPreparing {index + 1}/{len(selected_df)}: "
        f"{candidate_id}"
    )

    try:

        export_image, region, epsg, crs_transform = (
            build_export_image(row)
        )

        task = ee.batch.Export.image.toDrive(
            image=export_image,
            description=candidate_id,
            folder=DRIVE_FOLDER,
            fileNamePrefix=candidate_id,
            region=region,
            crs=epsg,
            crsTransform=crs_transform,
            maxPixels=100000000,
            fileFormat="GeoTIFF",
            formatOptions={
                "cloudOptimized": True
            }
        )

        export_records.append({
            "candidate_id": candidate_id,
            "city": city,
            "latitude": float(row["latitude"]),
            "longitude": float(row["longitude"]),
            "utm_epsg": epsg,
            "task": task
        })

    except Exception as error:

        print(
            "Could not prepare export:",
            candidate_id,
            error
        )


# Submit in batches.
print("\nSubmitting export tasks...")

submitted_records = []

for start in range(
    0, len(export_records), EXPORT_BATCH_SIZE
):

    batch = export_records[
        start:start + EXPORT_BATCH_SIZE
    ]

    print(
        f"\nSubmitting batch "
        f"{start // EXPORT_BATCH_SIZE + 1}"
    )

    for record in batch:

        try:

            record["task"].start()

            record["task_id"] = (
                record["task"].id
            )

            record["status"] = "SUBMITTED"

            submitted_records.append(record)

            print(
                "Started:",
                record["candidate_id"],
                record["task_id"]
            )

        except Exception as error:

            record["status"] = "SUBMISSION_FAILED"
            record["error"] = str(error)

            submitted_records.append(record)

            print(
                "Submission failed:",
                record["candidate_id"],
                error
            )

    # Small pause between batches.
    if start + EXPORT_BATCH_SIZE < len(export_records):
        time.sleep(5)


# =============================================================================
# 13. SAVE EXPORT MANIFEST
# =============================================================================

manifest_rows = []

for record in submitted_records:

    manifest_rows.append({
        "candidate_id": record["candidate_id"],
        "city": record["city"],
        "latitude": record["latitude"],
        "longitude": record["longitude"],
        "utm_epsg": record["utm_epsg"],
        "task_id": record.get("task_id", ""),
        "status": record.get("status", ""),
        "error": record.get("error", "")
    })

export_manifest = pd.DataFrame(
    manifest_rows
)

manifest_path = (
    OUTPUT_DIR / "exp20_export_manifest.csv"
)

export_manifest.to_csv(
    manifest_path,
    index=False
)


# =============================================================================
# 14. FINAL SUMMARY
# =============================================================================

print("\n" + "=" * 85)
print("EXPERIMENT 20 EXPORT SUBMISSION SUMMARY")
print("=" * 85)

print("Candidates generated:", len(candidates))
print("Candidates screened:", len(screened_df))
print("Tiles selected:", len(selected_df))
print("Export tasks prepared:", len(export_records))

print(
    "Tasks submitted:",
    sum(
        r.get("status") == "SUBMITTED"
        for r in submitted_records
    )
)

print(
    "Submission failures:",
    sum(
        r.get("status") == "SUBMISSION_FAILED"
        for r in submitted_records
    )
)

print("\nOutput folder:", OUTPUT_DIR)

print("\nFiles created:")
print(" - exp20_candidate_locations.csv")
print(" - exp20_screened_candidates.csv")
print(" - exp20_selected_tiles.csv")
print(" - exp20_export_manifest.csv")

print("\nGoogle Drive export folder:")
print(DRIVE_FOLDER)

print("\nIMPORTANT:")
print("1. Wait for all Earth Engine tasks to complete.")
print("2. Download the exported GeoTIFF files from Google Drive.")
print("3. Check that each file has 25 bands and a 256 x 256 grid.")
print("4. Convert SAR -9999 values into zero-filled data plus validity masks.")
print("5. Convert the label band to uint8.")
print("6. Keep all new tiles in TRAIN only.")
print("7. Recalculate class proportions from the actual preprocessed masks.")

print("\nExperiment 20 submission stage finished.")


EXPERIMENT 20 — REAL GEOGRAPHIC DATA COLLECTION
Project: /home/jovyan/vault/LULC_Project
Metadata: /home/jovyan/vault/LULC_Project/data/metadata
Output: /home/jovyan/vault/LULC_Project/exp20_new_real_tiles
Tile size: 256 x 256
Resolution: 30 m
Tile width: 7.68 km

Earth Engine initialized: final-project-513714

Metadata loaded successfully.
Training cities: 32
Original candidate centres: 500

Training cities:
['Berlin', 'Munich', 'Mannheim', 'Paris', 'Madrid', 'Rome', 'Warsaw', 'London', 'Stockholm', 'Athens', 'Istanbul', 'Cairo', 'Lagos', 'Nairobi', 'Johannesburg', 'Cape Town', 'Delhi', 'Mumbai', 'Bengaluru', 'Pune', 'Dhaka', 'Bangkok', 'Jakarta', 'Manila', 'Shanghai', 'Beijing', 'Tokyo', 'Seoul', 'Sydney', 'Melbourne', 'Lisbon', 'Amsterdam']

Generating candidate locations...
Berlin                20/30 candidates
Munich                23/30 candidates
Mannheim              23/30 candidates
Paris                 21/30 candidates
Madrid                21/30 candidates
Rome            

In [4]:
import ee
import pandas as pd

ee.Initialize(project="final-project-513714")

tasks = ee.batch.Task.list()

exp20_tasks = [
    task for task in tasks
    if task.config.get("description", "").startswith("E20_")
]

print("Experiment 20 tasks:", len(exp20_tasks))

for task in exp20_tasks:
    status = task.status()
    print(
        status.get("description"),
        ":", status.get("state"),
        status.get("error_message", "")
    )

from collections import Counter

print("\nSUMMARY:")
print(Counter(task.status().get("state") for task in exp20_tasks))

Experiment 20 tasks: 69
E20_Sydney_017 : READY 
E20_Sydney_022 : READY 
E20_Beijing_007 : READY 
E20_Pune_008 : READY 
E20_Cape Town_018 : READY 
E20_Seoul_019 : READY 
E20_Pune_001 : READY 
E20_Istanbul_010 : READY 
E20_Seoul_009 : READY 
E20_Mannheim_011 : READY 
E20_Bangkok_001 : READY 
E20_Pune_012 : READY 
E20_Manila_018 : READY 
E20_Pune_009 : READY 
E20_Seoul_012 : READY 
E20_Bangkok_017 : READY 
E20_Mumbai_003 : READY 
E20_Seoul_014 : READY 
E20_Shanghai_003 : READY 
E20_Shanghai_006 : READY 
E20_Johannesburg_011 : READY 
E20_Pune_003 : READY 
E20_Shanghai_013 : READY 
E20_Manila_012 : READY 
E20_Cape Town_020 : READY 
E20_Tokyo_012 : READY 
E20_Bengaluru_022 : READY 
E20_Shanghai_008 : READY 
E20_Tokyo_010 : READY 
E20_Mannheim_009 : READY 
E20_Tokyo_014 : READY 
E20_Johannesburg_001 : READY 
E20_Tokyo_022 : READY 
E20_Jakarta_011 : READY 
E20_Johannesburg_007 : READY 
E20_Bengaluru_013 : READY 
E20_Tokyo_018 : READY 
E20_Nairobi_019 : READY 
E20_Bengaluru_008 : READY 
E20_Jak

In [5]:

from pathlib import Path
from collections import Counter
import pandas as pd
import ee

# ---------------------------------------------------------
# PROJECT PATHS
# ---------------------------------------------------------

PROJECT = Path("/home/jovyan/vault/LULC_Project")

DATASETS = {
    "Original dataset": PROJECT / "processed_dataset_complete",
    "First expanded dataset": PROJECT / "processed_dataset_final_expanded",
    "Experiment 20 exports": PROJECT / "exp20_new_real_tiles",
}

# ---------------------------------------------------------
# 1. CHECK DATASET FOLDERS
# ---------------------------------------------------------

print("=" * 75)
print("DATASET FOLDER AUDIT")
print("=" * 75)

for name, path in DATASETS.items():

    print(f"\n{name}")
    print("-" * 60)
    print("Path:", path)
    print("Exists:", path.exists())

    if not path.exists():
        continue

    for split in ["train", "val", "test"]:

        split_path = path / split

        if not split_path.exists():
            continue

        print(f"\n  {split.upper()}")

        for category in ["images", "masks", "validity"]:

            category_path = split_path / category

            if category_path.exists():

                files = [
                    f for f in category_path.iterdir()
                    if f.is_file()
                ]

                print(
                    f"    {category:10s}: "
                    f"{len(files)} files"
                )

            else:
                print(f"    {category:10s}: folder missing")

# ---------------------------------------------------------
# 2. CHECK EXPERIMENT 20 EXPORT MANIFEST
# ---------------------------------------------------------

print("\n" + "=" * 75)
print("EXPERIMENT 20 EXPORT MANIFEST")
print("=" * 75)

manifest_path = (
    PROJECT /
    "exp20_new_real_tiles" /
    "exp20_export_manifest.csv"
)

if manifest_path.exists():

    manifest = pd.read_csv(manifest_path)

    print("Manifest rows:", len(manifest))

    if "status" in manifest.columns:
        print("\nSubmission status:")
        print(manifest["status"].value_counts(dropna=False))

    print("\nFirst five export records:")
    print(manifest.head().to_string(index=False))

else:
    print("Export manifest not found.")

# ---------------------------------------------------------
# 3. CHECK EARTH ENGINE TASK COMPLETION
# ---------------------------------------------------------

print("\n" + "=" * 75)
print("EARTH ENGINE EXPERIMENT 20 TASK STATUS")
print("=" * 75)

try:

    ee.Initialize(project="final-project-513714")

    tasks = ee.batch.Task.list()

    exp20_tasks = []

    for task in tasks:

        status = task.status()

        description = status.get("description", "")

        if description.startswith("E20_"):
            exp20_tasks.append(status)

    print("Experiment 20 tasks found:", len(exp20_tasks))

    states = Counter(
        task.get("state", "UNKNOWN")
        for task in exp20_tasks
    )

    print("\nTask summary:")
    for state, count in states.items():
        print(f"  {state}: {count}")

    if exp20_tasks:

        task_df = pd.DataFrame([
            {
                "description": t.get("description"),
                "state": t.get("state"),
                "error": t.get("error_message", "")
            }
            for t in exp20_tasks
        ])

        print("\nIndividual tasks:")
        print(task_df.to_string(index=False))

except Exception as error:

    print("Could not check Earth Engine tasks:", error)

# ---------------------------------------------------------
# 4. CHECK PROJECT METADATA
# ---------------------------------------------------------

print("\n" + "=" * 75)
print("PROJECT METADATA")
print("=" * 75)

metadata = PROJECT / "data" / "metadata"

for filename in [
    "sites.csv",
    "tile_manifest_candidates.csv"
]:

    path = metadata / filename

    print(
        filename,
        "FOUND" if path.exists() else "MISSING"
    )

    if path.exists():

        df = pd.read_csv(path)

        print("  Rows:", len(df))
        print("  Columns:", list(df.columns))

print("\n" + "=" * 75)
print("AUDIT FINISHED — NO DATA WAS MODIFIED")
print("=" * 75)


DATASET FOLDER AUDIT

Original dataset
------------------------------------------------------------
Path: /home/jovyan/vault/LULC_Project/processed_dataset_complete
Exists: True

  TRAIN
    images    : 320 files
    masks     : 320 files
    validity  : 320 files

  VAL
    images    : 80 files
    masks     : 80 files
    validity  : 80 files

  TEST
    images    : 100 files
    masks     : 100 files
    validity  : 100 files

First expanded dataset
------------------------------------------------------------
Path: /home/jovyan/vault/LULC_Project/processed_dataset_final_expanded
Exists: True

  TRAIN
    images    : 359 files
    masks     : 359 files
    validity  : 359 files

  VAL
    images    : 80 files
    masks     : 80 files
    validity  : 80 files

  TEST
    images    : 100 files
    masks     : 100 files
    validity  : 100 files

Experiment 20 exports
------------------------------------------------------------
Path: /home/jovyan/vault/LULC_Project/exp20_new_real_tiles


In [6]:

import ee
import time
import pandas as pd
from pathlib import Path
from datetime import datetime
from collections import Counter

# Initialize Earth Engine
ee.Initialize(project="final-project-513714")

PROJECT = Path("/home/jovyan/vault/LULC_Project")
OUTPUT = PROJECT / "exp20_new_real_tiles"

MANIFEST = OUTPUT / "exp20_export_manifest.csv"
LOG_FILE = OUTPUT / "exp20_live_tracking_log.csv"

# Read the 69 submitted task IDs
manifest = pd.read_csv(MANIFEST)

task_ids = set(
    manifest["task_id"].dropna().astype(str)
)

print("=" * 75)
print("EXPERIMENT 20 — LIVE EXPORT TRACKER")
print("=" * 75)
print("Tasks being monitored:", len(task_ids))
print("Refresh interval: 30 seconds")
print("Tracking log:", LOG_FILE)
print("\nPress the notebook stop button to pause monitoring.")
print("=" * 75)

previous_states = {}
log_records = []

try:
    while True:

        all_tasks = ee.batch.Task.list()

        matching_tasks = [
            task for task in all_tasks
            if task.id in task_ids
        ]

        statuses = []

        for task in matching_tasks:

            status = task.status()

            description = status.get(
                "description", task.id
            )

            state = status.get(
                "state", "UNKNOWN"
            )

            statuses.append({
                "task_id": task.id,
                "tile": description,
                "state": state,
                "error": status.get("error_message", "")
            })

            if previous_states.get(task.id) != state:

                timestamp = datetime.now().strftime(
                    "%Y-%m-%d %H:%M:%S"
                )

                print(
                    f"[{timestamp}] "
                    f"{description}: "
                    f"{previous_states.get(task.id, 'NEW')} "
                    f"-> {state}"
                )

                previous_states[task.id] = state

                log_records.append({
                    "timestamp": timestamp,
                    "tile": description,
                    "task_id": task.id,
                    "state": state,
                    "error": status.get("error_message", "")
                })

        counts = Counter(
            item["state"] for item in statuses
        )

        completed = counts.get("COMPLETED", 0)
        running = counts.get("RUNNING", 0)
        ready = counts.get("READY", 0)
        failed = counts.get("FAILED", 0)
        cancelled = counts.get("CANCELLED", 0)

        print("\n" + "-" * 75)
        print("TIME:", datetime.now().strftime("%H:%M:%S"))
        print("FOUND:", len(statuses), "/", len(task_ids))
        print("COMPLETED:", completed)
        print("RUNNING:", running)
        print("READY:", ready)
        print("FAILED:", failed)
        print("CANCELLED:", cancelled)

        active = [
            item for item in statuses
            if item["state"] == "RUNNING"
        ]

        if active:
            print("\nCURRENTLY EXPORTING:")

            for item in active:
                print(" ", item["tile"])

        errors = [
            item for item in statuses
            if item["state"] == "FAILED"
        ]

        if errors:
            print("\nFAILED TASKS:")

            for item in errors:
                print(item["tile"], ":", item["error"])

        # Save the transition log
        if log_records:
            pd.DataFrame(log_records).to_csv(
                LOG_FILE,
                index=False
            )

        print("-" * 75)

        terminal_states = {
            "COMPLETED", "FAILED", "CANCELLED"
        }

        if (
            len(statuses) == len(task_ids)
            and all(
                item["state"] in terminal_states
                for item in statuses
            )
        ):
            print("\nALL EXPERIMENT 20 TASKS HAVE FINISHED.")
            break

        time.sleep(30)

except KeyboardInterrupt:
    print("\nTracker paused by you.")
    print("You can run this cell again later.")


EXPERIMENT 20 — LIVE EXPORT TRACKER
Tasks being monitored: 69
Refresh interval: 30 seconds
Tracking log: /home/jovyan/vault/LULC_Project/exp20_new_real_tiles/exp20_live_tracking_log.csv

Press the notebook stop button to pause monitoring.
[2026-11-08 17:09:48] E20_Sydney_017: NEW -> READY
[2026-11-08 17:09:48] E20_Sydney_022: NEW -> READY
[2026-11-08 17:09:48] E20_Beijing_007: NEW -> READY
[2026-11-08 17:09:49] E20_Pune_008: NEW -> READY
[2026-11-08 17:09:49] E20_Cape Town_018: NEW -> READY
[2026-11-08 17:09:49] E20_Seoul_019: NEW -> READY
[2026-11-08 17:09:49] E20_Pune_001: NEW -> READY
[2026-11-08 17:09:50] E20_Istanbul_010: NEW -> READY
[2026-11-08 17:09:51] E20_Seoul_009: NEW -> READY
[2026-11-08 17:09:51] E20_Mannheim_011: NEW -> READY
[2026-11-08 17:09:52] E20_Bangkok_001: NEW -> READY
[2026-11-08 17:09:52] E20_Pune_012: NEW -> READY
[2026-11-08 17:09:52] E20_Manila_018: NEW -> READY
[2026-11-08 17:09:52] E20_Pune_009: NEW -> READY
[2026-11-08 17:09:53] E20_Seoul_012: NEW -> READ

In [7]:

from pathlib import Path
import pandas as pd

PROJECT = Path("/home/jovyan/vault/LULC_Project")

E20_DIR = PROJECT / "exp20_new_real_tiles"
EXPANDED = PROJECT / "processed_dataset_final_expanded"
UNIFIED = PROJECT / "processed_dataset_unified"

print("=" * 70)
print("FINAL DATASET COMBINATION READINESS CHECK")
print("=" * 70)

# 1. Check export manifest
manifest_path = E20_DIR / "exp20_export_manifest.csv"

if manifest_path.exists():
    manifest = pd.read_csv(manifest_path)

    print("\n1. EARTH ENGINE EXPORTS")

    if "status" in manifest.columns:
        print(manifest["status"].value_counts())

# 2. Find downloaded GeoTIFF files
tif_files = list(E20_DIR.rglob("*.tif")) + list(E20_DIR.rglob("*.tiff"))

print("\n2. DOWNLOADED GEOTIFF FILES")
print("Found:", len(tif_files))

# 3. Check whether all 69 tile IDs have a local GeoTIFF
if manifest_path.exists():
    expected_ids = set(manifest["candidate_id"].astype(str))
    downloaded_ids = {p.stem for p in tif_files}

    matched = expected_ids & downloaded_ids

    print("Expected:", len(expected_ids))
    print("Matched locally:", len(matched))
    print("Not found locally:", len(expected_ids - downloaded_ids))

    if expected_ids - downloaded_ids:
        print("\nExamples not found:")
        print(sorted(expected_ids - downloaded_ids)[:15])

# 4. Check existing expanded dataset
print("\n3. FIRST EXPANDED DATASET")

for split in ["train", "val", "test"]:
    print(f"\n{split.upper()}")

    for folder in ["images", "masks", "validity"]:
        path = EXPANDED / split / folder

        count = len(list(path.glob("*"))) if path.exists() else 0

        print(f"  {folder}: {count}")

# 5. Check whether a unified dataset already exists
print("\n4. UNIFIED DATASET")

print("Exists:", UNIFIED.exists())

if UNIFIED.exists():
    for split in ["train", "val", "test"]:
        for folder in ["images", "masks", "validity"]:
            path = UNIFIED / split / folder
            count = len(list(path.glob("*"))) if path.exists() else 0
            print(f"{split}/{folder}: {count}")

print("\n" + "=" * 70)
print("CHECK COMPLETE — NO FILES MODIFIED")
print("=" * 70)


FINAL DATASET COMBINATION READINESS CHECK

1. EARTH ENGINE EXPORTS
status
SUBMITTED    69
Name: count, dtype: int64

2. DOWNLOADED GEOTIFF FILES
Found: 0
Expected: 69
Matched locally: 0
Not found locally: 69

Examples not found:
['E20_Bangkok_001', 'E20_Bangkok_017', 'E20_Beijing_007', 'E20_Bengaluru_003', 'E20_Bengaluru_008', 'E20_Bengaluru_013', 'E20_Bengaluru_018', 'E20_Bengaluru_022', 'E20_Cairo_002', 'E20_Cairo_013', 'E20_Cairo_014', 'E20_Cairo_021', 'E20_Cairo_023', 'E20_Cape Town_010', 'E20_Cape Town_018']

3. FIRST EXPANDED DATASET

TRAIN
  images: 359
  masks: 359
  validity: 359

VAL
  images: 80
  masks: 80
  validity: 80

TEST
  images: 100
  masks: 100
  validity: 100

4. UNIFIED DATASET
Exists: False

CHECK COMPLETE — NO FILES MODIFIED


In [8]:
import ee
import pandas as pd
from pathlib import Path
from collections import Counter

PROJECT_DIR = Path("/home/jovyan/vault/LULC_Project")
MANIFEST = PROJECT_DIR / "exp20_new_real_tiles" / "exp20_export_manifest.csv"

# Initialize Earth Engine
ee.Initialize(project="final-project-513714")

# Read the export manifest
df = pd.read_csv(MANIFEST)

print("Manifest columns:", df.columns.tolist())

# Retrieve live Earth Engine task statuses
tasks = ee.batch.Task.list()

live_status = {}
for task in tasks:
    try:
        s = task.status()
        task_id = s.get("id")
        if task_id:
            live_status[str(task_id)] = {
                "state": s.get("state", "UNKNOWN"),
                "description": s.get("description", "")
            }
    except Exception:
        pass

# Identify the task ID column
id_candidates = ["task_id", "taskId", "id", "task"]
id_col = next(
    (c for c in id_candidates if c in df.columns),
    None
)

if id_col is None:
    print("\nCould not identify a task ID column.")
    print("Manifest preview:")
    display(df.head())
    print("\nLive task states:")
    print(Counter(v["state"] for v in live_status.values()))
else:
    df["live_state"] = df[id_col].astype(str).map(
        lambda x: live_status.get(x, {}).get("state", "NOT_IN_RECENT_TASK_LIST")
    )

    print("\nLIVE EXPORT STATUS")
    print(df["live_state"].value_counts(dropna=False))

    print("\nFAILED / CANCELLED TASKS")
    display(df[df["live_state"].isin(["FAILED", "CANCELLED"])])

    print("\nTASKS NOT YET COMPLETED")
    display(df[df["live_state"] != "COMPLETED"])

    print("\nCOMPLETED:", (df["live_state"] == "COMPLETED").sum())
    print("TOTAL IN MANIFEST:", len(df))

Manifest columns: ['candidate_id', 'city', 'latitude', 'longitude', 'utm_epsg', 'task_id', 'status', 'error']

LIVE EXPORT STATUS
live_state
COMPLETED    69
Name: count, dtype: int64

FAILED / CANCELLED TASKS


,candidate_id,city,latitude,longitude,utm_epsg,task_id,status,error,live_state



TASKS NOT YET COMPLETED


,candidate_id,city,latitude,longitude,utm_epsg,task_id,status,error,live_state



COMPLETED: 69
TOTAL IN MANIFEST: 69


In [10]:
from pathlib import Path

SEARCH_DIRS = [
    Path("/content/drive/MyDrive"),
    Path("/content/drive/Shareddrives"),
    Path("/home/jovyan/drive"),
    Path("/home/jovyan/Drive"),
    Path("/home/jovyan/MyDrive"),
    Path("/home/jovyan/vault"),
]

print("=" * 65)
print("SEARCHING FOR EXPERIMENT 20 EXPORTS")
print("=" * 65)

for root in SEARCH_DIRS:
    if not root.exists():
        print(f"NOT MOUNTED: {root}")
        continue

    print(f"\nFOUND DIRECTORY: {root}")

    matches = list(root.rglob("E20_*.tif")) + list(root.rglob("E20_*.tiff"))

    print("Experiment 20 GeoTIFFs found:", len(matches))

    if matches:
        for p in matches[:10]:
            print(" ", p)

SEARCHING FOR EXPERIMENT 20 EXPORTS
NOT MOUNTED: /content/drive/MyDrive
NOT MOUNTED: /content/drive/Shareddrives
NOT MOUNTED: /home/jovyan/drive
NOT MOUNTED: /home/jovyan/Drive
NOT MOUNTED: /home/jovyan/MyDrive

FOUND DIRECTORY: /home/jovyan/vault
Experiment 20 GeoTIFFs found: 0


In [11]:

# ================================================================
# EXPERIMENT 20 — DIRECT EARTH ENGINE DOWNLOAD
# No Google Drive required
# ================================================================

!pip -q install earthengine-api rasterio requests tqdm

import ee
import pandas as pd
import numpy as np
import requests
import rasterio
import time
from pathlib import Path
from tqdm.auto import tqdm

# ------------------------------------------------
# PATHS
# ------------------------------------------------

PROJECT = Path("/home/jovyan/vault/LULC_Project")

MANIFEST = (
    PROJECT / "exp20_new_real_tiles" /
    "exp20_export_manifest.csv"
)

RAW_DIR = (
    PROJECT / "exp20_new_real_tiles" /
    "direct_download"
)

RAW_DIR.mkdir(parents=True, exist_ok=True)

assert MANIFEST.exists(), f"Missing manifest: {MANIFEST}"

# ------------------------------------------------
# EARTH ENGINE
# ------------------------------------------------

try:
    ee.Initialize(project="final-project-513714")
except Exception:
    ee.Authenticate()
    ee.Initialize(project="final-project-513714")

# ------------------------------------------------
# READ THE SAME 69 LOCATIONS
# ------------------------------------------------

df = pd.read_csv(MANIFEST)

required = [
    "candidate_id", "city", "latitude",
    "longitude", "utm_epsg"
]

for col in required:
    assert col in df.columns, f"Missing column: {col}"

assert len(df) == 69, f"Expected 69 candidates, found {len(df)}"

# ------------------------------------------------
# DATA SOURCES AND CLASS MAPPING
# ------------------------------------------------

S1 = ee.ImageCollection("COPERNICUS/S1_GRD")
WC = ee.Image("ESA/WorldCover/v200/2021")

WORLD_COVER_VALUES = [
    10, 20, 30, 40, 50, 60,
    70, 80, 90, 95, 100
]

MERGED_CLASSES = [
    0, 1, 1, 2, 3, 4,
    4, 5, 5, 5, 1
]

# ------------------------------------------------
# BUILD ONE TILE
# ------------------------------------------------

def build_tile(row):

    lat = float(row.latitude)
    lon = float(row.longitude)
    epsg = int(row.utm_epsg)

    projection = ee.Projection(f"EPSG:{epsg}")

    center = ee.Geometry.Point([lon, lat]).transform(
        projection, 1
    )

    coords = center.coordinates()

    # 256 pixels x 30 metres = 7680 metres
    half = 3840

    xmin = ee.Number(coords.get(0)).subtract(half)
    xmax = ee.Number(coords.get(0)).add(half)
    ymin = ee.Number(coords.get(1)).subtract(half)
    ymax = ee.Number(coords.get(1)).add(half)

    region = ee.Geometry.Rectangle(
        [xmin, ymin, xmax, ymax],
        proj=projection,
        geodesic=False
    )

    monthly_bands = []

    # 12 months of VV/VH
    for month in range(1, 13):

        start = ee.Date.fromYMD(2021, month, 1)
        end = start.advance(1, "month")

        collection = (
            S1.filterBounds(region)
              .filterDate(start, end)
              .filter(ee.Filter.eq("instrumentMode", "IW"))
              .filter(
                  ee.Filter.listContains(
                      "transmitterReceiverPolarisation", "VV"
                  )
              )
              .filter(
                  ee.Filter.listContains(
                      "transmitterReceiverPolarisation", "VH"
                  )
              )
              .select(["VV", "VH"])
        )

        # Monthly median, or -9999 when no acquisition exists
        monthly = ee.Image(
            ee.Algorithms.If(
                collection.size().gt(0),
                collection.median().select(["VV", "VH"]),
                ee.Image.constant([-9999, -9999])
                    .rename(["VV", "VH"])
            )
        )

        monthly = monthly.rename([
            f"VV_{month:02d}",
            f"VH_{month:02d}"
        ])

        monthly_bands.append(monthly)

    sar = ee.Image.cat(monthly_bands)

    # WorldCover -> six merged classes
    labels = (
        WC.select("Map")
          .remap(WORLD_COVER_VALUES, MERGED_CLASSES)
          .rename("label")
          .toFloat()
    )

    image = sar.addBands(labels).clip(region)

    # Match the 256 x 256, 30m UTM export grid
    transform = [
        30, 0, float(coords.getInfo()[0]) - half,
        0, -30, float(coords.getInfo()[1]) + half
    ]

    return image, region, transform, f"EPSG:{epsg}"


# ------------------------------------------------
# DOWNLOAD WITH RETRIES
# ------------------------------------------------

def download_tile(row):

    candidate_id = str(row.candidate_id)
    output = RAW_DIR / f"{candidate_id}.tif"

    if output.exists() and output.stat().st_size > 1000:
        try:
            with rasterio.open(output) as src:
                if src.count == 25 and src.width == 256 and src.height == 256:
                    return "ALREADY_DOWNLOADED"
        except Exception:
            pass

        output.unlink()

    image, region, transform, crs = build_tile(row)

    params = {
        "name": candidate_id,
        "crs": crs,
        "crs_transform": transform,
        "region": region,
        "format": "GEO_TIFF"
    }

    for attempt in range(1, 5):
        try:
            url = image.getDownloadURL(params)

            response = requests.get(
                url,
                timeout=180
            )
            response.raise_for_status()

            output.write_bytes(response.content)

            # Validate the downloaded GeoTIFF
            with rasterio.open(output) as src:
                assert src.count == 25, (
                    f"{candidate_id}: expected 25 bands, got {src.count}"
                )
                assert src.width == 256 and src.height == 256, (
                    f"{candidate_id}: unexpected dimensions"
                )

            return "SUCCESS"

        except Exception as exc:
            if output.exists():
                output.unlink()

            if attempt == 4:
                return f"FAILED: {str(exc)[:250]}"

            time.sleep(attempt * 5)


# ------------------------------------------------
# DOWNLOAD ALL 69
# ------------------------------------------------

results = []

for _, row in tqdm(
    df.iterrows(),
    total=len(df),
    desc="Downloading Experiment 20"
):
    status = download_tile(row)

    results.append({
        "candidate_id": row.candidate_id,
        "city": row.city,
        "download_status": status
    })

    print(row.candidate_id, status)

# ------------------------------------------------
# FINAL AUDIT
# ------------------------------------------------

audit = pd.DataFrame(results)

audit_path = RAW_DIR / "direct_download_audit.csv"
audit.to_csv(audit_path, index=False)

print("\n" + "=" * 65)
print("EXPERIMENT 20 DIRECT DOWNLOAD SUMMARY")
print("=" * 65)

print(audit["download_status"].value_counts())

success = (audit["download_status"] == "SUCCESS").sum()
already = (audit["download_status"] == "ALREADY_DOWNLOADED").sum()
failed = len(audit) - success - already

print("\nNewly downloaded:", success)
print("Already downloaded:", already)
print("Failed:", failed)
print("Total available:", success + already)

print("\nRaw files saved to:")
print(RAW_DIR)



[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: python3 -m pip install --upgrade pip


ValueError: invalid literal for int() with base 10: 'EPSG:32636'

In [13]:

# =============================================================================
# EXPERIMENT 20 — DIRECT EARTH ENGINE DOWNLOAD + DIMENSION FIX + DATASET MERGE
#
# Base dataset:
#   processed_dataset_final_expanded
#   359 train + 80 val + 100 test
#
# New E20 tiles:
#   69 selected geographic training tiles
#
# Final dataset:
#   processed_dataset_unified
#   428 train + 80 val + 100 test
#
# Important:
#   - Downloads directly from Earth Engine.
#   - Does NOT use Google Drive.
#   - Reuses valid existing downloads.
#   - Crops only extra bottom/right pixels after checking alignment.
#   - Does NOT resize or interpolate SAR data.
#   - Does NOT modify the original expanded dataset.
#   - Adds new tiles to TRAIN only.
# =============================================================================


# =============================================================================
# 1. INSTALL AND IMPORT PACKAGES
# =============================================================================

import sys
import subprocess
import importlib.util

required_packages = {
    "earthengine-api": "ee",
    "rasterio": "rasterio",
    "requests": "requests",
    "pandas": "pandas",
    "numpy": "numpy"
}

for package, module in required_packages.items():
    if importlib.util.find_spec(module) is None:
        print(f"Installing {package}...")
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", package
        ])

import os
import re
import io
import json
import time
import shutil
import zipfile
import warnings
from pathlib import Path

import ee
import rasterio
import requests
import numpy as np
import pandas as pd

from rasterio.transform import Affine

warnings.filterwarnings("ignore")


# =============================================================================
# 2. CONFIGURATION
# =============================================================================

PROJECT_DIR = Path("/home/jovyan/vault/LULC_Project")

BASE_DATASET = (
    PROJECT_DIR / "processed_dataset_final_expanded"
)

SELECTED_CSV = (
    PROJECT_DIR
    / "exp20_new_real_tiles"
    / "exp20_selected_tiles.csv"
)

RAW_DOWNLOAD_DIR = (
    PROJECT_DIR
    / "exp20_new_real_tiles"
    / "direct_download"
)

FINAL_DATASET = (
    PROJECT_DIR / "processed_dataset_unified"
)

EE_PROJECT = "final-project-513714"

TILE_SIZE = 256
RESOLUTION = 30
HALF_TILE_M = TILE_SIZE * RESOLUTION / 2

EXPECTED_BANDS = 25
EXPECTED_HEIGHT = 256
EXPECTED_WIDTH = 256

EXPECTED_NEW_TILES = 69

CLASS_NAMES = [
    "Tree",
    "Shrub_Grass_Moss",
    "Cropland",
    "Built_up",
    "Bare_Snow_Ice",
    "Water_Wetland_Mangrove"
]

NUM_CLASSES = 6

# Training-only robust statistics from the existing project.
# This follows the robust normalization used in the previous pipeline.
VV_MEDIAN = -10.2969
VV_IQR = 5.75

VH_MEDIAN = -16.9375
VH_IQR = 6.7656

# Sentinel-1 missing-data threshold.
NODATA_VALUE = -9999

RAW_DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 85)
print("EXPERIMENT 20 — DIRECT DOWNLOAD AND DATASET MERGE")
print("=" * 85)

print("Base dataset:", BASE_DATASET)
print("Selected CSV:", SELECTED_CSV)
print("Raw download folder:", RAW_DOWNLOAD_DIR)
print("Final dataset:", FINAL_DATASET)


# =============================================================================
# 3. CHECK INPUTS
# =============================================================================

if not BASE_DATASET.exists():
    raise FileNotFoundError(
        f"Base dataset not found:\n{BASE_DATASET}"
    )

if not SELECTED_CSV.exists():
    raise FileNotFoundError(
        f"Selected tiles CSV not found:\n{SELECTED_CSV}"
    )

selected_df = pd.read_csv(SELECTED_CSV)

required_columns = {
    "candidate_id",
    "city",
    "latitude",
    "longitude",
    "utm_epsg"
}

missing_columns = required_columns - set(selected_df.columns)

if missing_columns:
    raise ValueError(
        f"Selected CSV is missing columns: {missing_columns}"
    )

selected_df["candidate_id"] = (
    selected_df["candidate_id"].astype(str)
)

if selected_df["candidate_id"].duplicated().any():
    raise ValueError(
        "Duplicate candidate_id values found in selected CSV."
    )

print("\nSelected tiles:", len(selected_df))

if len(selected_df) != EXPECTED_NEW_TILES:
    raise ValueError(
        f"Expected {EXPECTED_NEW_TILES} selected tiles, "
        f"but found {len(selected_df)}."
    )


# =============================================================================
# 4. CHECK BASE DATASET STRUCTURE
# =============================================================================

for split in ["train", "val", "test"]:
    for folder in ["images", "masks", "validity"]:

        path = BASE_DATASET / folder / split

        if not path.exists():
            raise FileNotFoundError(
                f"Missing dataset folder: {path}"
            )


def get_npy_stems(folder):
    return {
        p.stem
        for p in folder.glob("*.npy")
    }


print("\nChecking original expanded dataset...")

base_counts = {}

for split in ["train", "val", "test"]:

    image_stems = get_npy_stems(
        BASE_DATASET / "images" / split
    )

    mask_stems = get_npy_stems(
        BASE_DATASET / "masks" / split
    )

    validity_stems = get_npy_stems(
        BASE_DATASET / "validity" / split
    )

    if not (
        image_stems == mask_stems == validity_stems
    ):
        raise RuntimeError(
            f"File mismatch in base {split} split.\n"
            f"Images: {len(image_stems)}\n"
            f"Masks: {len(mask_stems)}\n"
            f"Validity: {len(validity_stems)}"
        )

    base_counts[split] = len(image_stems)

    print(
        f"{split.upper():5s}: {len(image_stems)} tiles"
    )

if base_counts != {
    "train": 359,
    "val": 80,
    "test": 100
}:
    raise RuntimeError(
        "The expanded dataset does not have the expected "
        "359/80/100 split. Stopping to avoid an incorrect merge."
    )


# Inspect an existing image to establish expected array dimensions.
sample_image_path = next(
    (BASE_DATASET / "images" / "train").glob("*.npy")
)

sample_mask_path = next(
    (BASE_DATASET / "masks" / "train").glob("*.npy")
)

sample_validity_path = next(
    (BASE_DATASET / "validity" / "train").glob("*.npy")
)

sample_image = np.load(sample_image_path)
sample_mask = np.load(sample_mask_path)
sample_validity = np.load(sample_validity_path)

print("\nExisting sample shapes:")
print("Image:", sample_image.shape, sample_image.dtype)
print("Mask:", sample_mask.shape, sample_mask.dtype)
print("Validity:", sample_validity.shape, sample_validity.dtype)

if sample_image.shape != (12, 2, 256, 256):
    raise ValueError(
        "Existing image arrays are not shaped (12, 2, 256, 256). "
        "Do not merge until the preprocessing format is confirmed."
    )

if sample_mask.shape != (256, 256):
    raise ValueError(
        "Existing masks are not shaped (256, 256)."
    )

if sample_validity.shape not in [
    (12, 256, 256),
    (12, 2, 256, 256),
    (256, 256)
]:
    raise ValueError(
        f"Unexpected validity shape: {sample_validity.shape}"
    )


# =============================================================================
# 5. INITIALIZE EARTH ENGINE
# =============================================================================

try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)

print("\nEarth Engine initialized:", EE_PROJECT)


# =============================================================================
# 6. PREPARE SENTINEL-1 AND WORLDCOVER
# =============================================================================

START_DATE = "2021-01-01"
END_DATE = "2022-01-01"

s1 = (
    ee.ImageCollection("COPERNICUS/S1_GRD")
    .filterDate(START_DATE, END_DATE)
    .filter(ee.Filter.eq("instrumentMode", "IW"))
    .filter(
        ee.Filter.listContains(
            "transmitterReceiverPolarisation", "VV"
        )
    )
    .filter(
        ee.Filter.listContains(
            "transmitterReceiverPolarisation", "VH"
        )
    )
    .select(["VV", "VH"])
)

worldcover = (
    ee.ImageCollection("ESA/WorldCover/v200")
    .first()
    .select("Map")
)

worldcover_merged = worldcover.remap(
    [10, 20, 30, 40, 50, 60, 70, 80, 90, 95, 100],
    [0, 1, 1, 2, 3, 4, 4, 5, 5, 5, 1],
    255
).rename("label").toUint8()

print("Sentinel-1 and WorldCover prepared.")


# =============================================================================
# 7. BUILD ONE TILE'S 25-BAND EARTH ENGINE IMAGE
# =============================================================================

def parse_epsg(value):

    value = str(value).strip().upper()

    value = value.replace("EPSG:", "")

    return f"EPSG:{int(value)}"


def build_tile_image(row):

    lat = float(row["latitude"])
    lon = float(row["longitude"])

    epsg = parse_epsg(row["utm_epsg"])

    projection = ee.Projection(epsg)

    centre = (
        ee.Geometry.Point([lon, lat])
        .transform(projection, 1)
    )

    xy = centre.coordinates()

    x = ee.Number(xy.get(0))
    y = ee.Number(xy.get(1))

    xmin = x.subtract(HALF_TILE_M)
    xmax = x.add(HALF_TILE_M)

    ymin = y.subtract(HALF_TILE_M)
    ymax = y.add(HALF_TILE_M)

    region = ee.Geometry.Rectangle(
        [xmin, ymin, xmax, ymax],
        projection,
        False
    )

    monthly_images = []

    for month in range(1, 13):

        start = ee.Date.fromYMD(2021, month, 1)
        end = start.advance(1, "month")

        collection = (
            s1.filterDate(start, end)
            .filterBounds(region)
        )

        empty_image = (
            ee.Image.constant([-9999, -9999])
            .rename(["VV", "VH"])
            .toFloat()
        )

        monthly = ee.Image(
            ee.Algorithms.If(
                collection.size().gt(0),
                collection.median().select(["VV", "VH"]),
                empty_image
            )
        )

        monthly = monthly.rename([
            f"VV_{month:02d}",
            f"VH_{month:02d}"
        ])

        monthly_images.append(
            monthly.clip(region)
        )

    sar_stack = monthly_images[0]

    for monthly in monthly_images[1:]:
        sar_stack = sar_stack.addBands(monthly)

    label = (
        worldcover_merged
        .clip(region)
        .rename("label")
    )

    export_image = (
        sar_stack
        .addBands(label)
        .toFloat()
        .clip(region)
    )

    # Exact 30 m, 256 x 256 grid.
    crs_transform = [
        RESOLUTION, 0, xmin,
        0, -RESOLUTION, ymax
    ]

    return (
        export_image,
        region,
        epsg,
        crs_transform,
        xmin,
        ymax
    )


# =============================================================================
# 8. DOWNLOAD AND VALIDATE GEOTIFF
# =============================================================================

def safe_filename(candidate_id):

    return re.sub(
        r"[^A-Za-z0-9_.-]+",
        "_",
        str(candidate_id)
    )


def extract_geotiff(response_content, candidate_id):

    """
    Earth Engine may return a TIFF directly or a ZIP containing a TIFF.
    Save the extracted TIFF to a temporary file.
    """

    temp_path = RAW_DOWNLOAD_DIR / (
        safe_filename(candidate_id) + ".download"
    )

    with open(temp_path, "wb") as f:
        f.write(response_content)

    # ZIP response
    if zipfile.is_zipfile(temp_path):

        with zipfile.ZipFile(temp_path, "r") as zf:

            tif_names = [
                name for name in zf.namelist()
                if name.lower().endswith((".tif", ".tiff"))
            ]

            if len(tif_names) == 0:
                raise RuntimeError(
                    f"{candidate_id}: ZIP contains no GeoTIFF."
                )

            if len(tif_names) > 1:
                print(
                    f"Warning: {candidate_id} ZIP has multiple TIFFs; "
                    "using the first."
                )

            extracted = zf.read(tif_names[0])

        temp_path.unlink(missing_ok=True)

        final_raw_path = RAW_DOWNLOAD_DIR / (
            safe_filename(candidate_id) + ".tif"
        )

        with open(final_raw_path, "wb") as f:
            f.write(extracted)

        return final_raw_path

    # Direct TIFF response
    final_raw_path = RAW_DOWNLOAD_DIR / (
        safe_filename(candidate_id) + ".tif"
    )

    os.replace(temp_path, final_raw_path)

    return final_raw_path


def expected_transform_values(crs_transform):

    # Earth Engine crs_transform:
    # [x_scale, x_shear, x_origin, y_shear, y_scale, y_origin]

    return [
        float(crs_transform[0]),
        float(crs_transform[1]),
        float(crs_transform[2]),
        float(crs_transform[3]),
        float(crs_transform[4]),
        float(crs_transform[5])
    ]


def validate_and_read_geotiff(
    tif_path,
    candidate_id,
    crs_transform
):

    """
    Validate band count, dimensions and grid alignment.

    Accepts:
      256 x 256
      257 x 256
      256 x 257
      257 x 257

    Crops only the extra bottom/right pixels.

    Rejects:
      - shifted grids
      - unexpected dimensions
      - missing bands
      - dimensions smaller than 256
    """

    with rasterio.open(tif_path) as src:

        if src.count != EXPECTED_BANDS:
            raise ValueError(
                f"{candidate_id}: expected 25 bands, "
                f"received {src.count}."
            )

        if (
            src.height < EXPECTED_HEIGHT
            or src.width < EXPECTED_WIDTH
        ):
            raise ValueError(
                f"{candidate_id}: raster too small: "
                f"{src.width} x {src.height}"
            )

        if (
            src.height > EXPECTED_HEIGHT + 1
            or src.width > EXPECTED_WIDTH + 1
        ):
            raise ValueError(
                f"{candidate_id}: unexpected dimensions: "
                f"{src.width} x {src.height}"
            )

        # Check that the downloaded raster starts on the intended grid.
        actual = [
            src.transform.a,
            src.transform.b,
            src.transform.c,
            src.transform.d,
            src.transform.e,
            src.transform.f
        ]

        expected = expected_transform_values(crs_transform)

        if not np.allclose(
            actual,
            expected,
            atol=0.05,
            rtol=0
        ):
            raise ValueError(
                f"{candidate_id}: GeoTIFF transform does not match "
                "the requested 30 m grid.\n"
                f"Actual:   {actual}\n"
                f"Expected: {expected}\n"
                "Refusing to crop a potentially misaligned raster."
            )

        data = src.read(
            window=rasterio.windows.Window(
                col_off=0,
                row_off=0,
                width=EXPECTED_WIDTH,
                height=EXPECTED_HEIGHT
            )
        )

        if data.shape != (
            EXPECTED_BANDS,
            EXPECTED_HEIGHT,
            EXPECTED_WIDTH
        ):
            raise ValueError(
                f"{candidate_id}: incorrect cropped shape: {data.shape}"
            )

        return data


def download_one_tile(row):

    candidate_id = str(row["candidate_id"])

    tif_path = RAW_DOWNLOAD_DIR / (
        safe_filename(candidate_id) + ".tif"
    )

    export_image, region, epsg, crs_transform, xmin, ymax = (
        build_tile_image(row)
    )

    # Reuse an existing TIFF only if it passes all checks.
    if tif_path.exists():

        try:

            data = validate_and_read_geotiff(
                tif_path,
                candidate_id,
                crs_transform
            )

            return data, "REUSED"

        except Exception as error:

            print(
                f"Existing download invalid for {candidate_id}: {error}"
            )

            tif_path.unlink(missing_ok=True)

    params = {
        "name": safe_filename(candidate_id),
        "crs": epsg,
        "crs_transform": crs_transform,
        "region": region,
        "format": "GEO_TIFF"
    }

    last_error = None

    for attempt in range(1, 4):

        try:

            download_url = export_image.getDownloadURL(params)

            response = requests.get(
                download_url,
                timeout=180
            )

            response.raise_for_status()

            tif_path = extract_geotiff(
                response.content,
                candidate_id
            )

            data = validate_and_read_geotiff(
                tif_path,
                candidate_id,
                crs_transform
            )

            return data, "DOWNLOADED"

        except Exception as error:

            last_error = error

            print(
                f"Attempt {attempt}/3 failed for "
                f"{candidate_id}: {error}"
            )

            tif_path.unlink(missing_ok=True)

            if attempt < 3:
                time.sleep(5 * attempt)

    raise RuntimeError(
        f"{candidate_id}: failed after 3 attempts: {last_error}"
    )


# =============================================================================
# 9. PREPROCESS SAR AND LABEL ARRAYS
# =============================================================================

def preprocess_tile(raw_data, candidate_id):

    """
    Input:
      raw_data: (25, 256, 256)

    Bands:
      0,1 = January VV,VH
      2,3 = February VV,VH
      ...
      22,23 = December VV,VH
      24 = merged WorldCover label

    Output:
      image:    (12, 2, 256, 256) float32
      mask:     (256, 256) uint8
      validity: same shape as existing dataset validity array
    """

    if raw_data.shape != (25, 256, 256):
        raise ValueError(
            f"{candidate_id}: unexpected raw shape {raw_data.shape}"
        )

    raw_sar = raw_data[:24].astype(np.float32)

    raw_sar = raw_sar.reshape(
        12, 2, 256, 256
    )

    # SAR values must be finite and not nodata.
    sar_valid = (
        np.isfinite(raw_sar)
        & (raw_sar > -9998)
        & (raw_sar < 100)
    )

    # One monthly validity value per pixel:
    # both VV and VH must be valid.
    monthly_valid = sar_valid.all(axis=1)

    # Copy before replacing invalid values.
    image = raw_sar.copy()

    # VV normalization.
    vv = image[:, 0, :, :]

    vv_valid = sar_valid[:, 0, :, :]

    vv[~vv_valid] = VV_MEDIAN

    vv = (vv - VV_MEDIAN) / VV_IQR

    # VH normalization.
    vh = image[:, 1, :, :]

    vh_valid = sar_valid[:, 1, :, :]

    vh[~vh_valid] = VH_MEDIAN

    vh = (vh - VH_MEDIAN) / VH_IQR

    image[:, 0, :, :] = vv
    image[:, 1, :, :] = vh

    image = np.nan_to_num(
        image,
        nan=0.0,
        posinf=0.0,
        neginf=0.0
    ).astype(np.float32)

    # WorldCover merged labels.
    raw_label = raw_data[24]

    raw_label = np.rint(raw_label).astype(np.int32)

    valid_labels = np.isin(
        raw_label,
        np.arange(NUM_CLASSES)
    )

    mask = np.where(
        valid_labels,
        raw_label,
        255
    ).astype(np.uint8)

    # Match the existing dataset validity-array shape.
    if sample_validity.shape == (12, 256, 256):

        validity = monthly_valid.astype(np.uint8)

    elif sample_validity.shape == (12, 2, 256, 256):

        validity = sar_valid.astype(np.uint8)

    elif sample_validity.shape == (256, 256):

        validity = monthly_valid.any(axis=0).astype(np.uint8)

    else:
        raise ValueError(
            f"Unsupported existing validity shape: "
            f"{sample_validity.shape}"
        )

    if image.shape != sample_image.shape:
        raise ValueError(
            f"{candidate_id}: image shape mismatch."
        )

    if mask.shape != sample_mask.shape:
        raise ValueError(
            f"{candidate_id}: mask shape mismatch."
        )

    if validity.shape != sample_validity.shape:
        raise ValueError(
            f"{candidate_id}: validity shape mismatch."
        )

    return image, mask, validity


# =============================================================================
# 10. CREATE OR RESUME THE UNIFIED DATASET
# =============================================================================

if not FINAL_DATASET.exists():

    print("\nCreating a separate copy of the expanded dataset...")

    shutil.copytree(
        BASE_DATASET,
        FINAL_DATASET
    )

    print("Base dataset copied.")

else:

    print(
        "\nUnified dataset already exists. "
        "Checking it before continuing."
    )


# Check that all original base files are present in the unified copy.
for split in ["train", "val", "test"]:

    for folder in ["images", "masks", "validity"]:

        base_stems = get_npy_stems(
            BASE_DATASET / folder / split
        )

        final_stems = get_npy_stems(
            FINAL_DATASET / folder / split
        )

        missing = base_stems - final_stems

        if missing:
            raise RuntimeError(
                f"Unified dataset is missing {len(missing)} "
                f"original {folder}/{split} files. "
                "Stopping rather than silently producing an incomplete copy."
            )


# =============================================================================
# 11. DOWNLOAD, PREPROCESS AND SAVE THE 69 TILES
# =============================================================================

success_count = 0
reused_count = 0
failed_records = []

print("\n" + "=" * 85)
print("STARTING E20 DOWNLOAD AND PREPROCESSING")
print("=" * 85)

for index, row in selected_df.iterrows():

    candidate_id = str(row["candidate_id"])

    safe_id = safe_filename(candidate_id)

    image_path = (
        FINAL_DATASET / "images" / "train" / f"{safe_id}.npy"
    )

    mask_path = (
        FINAL_DATASET / "masks" / "train" / f"{safe_id}.npy"
    )

    validity_path = (
        FINAL_DATASET / "validity" / "train" / f"{safe_id}.npy"
    )

    print(
        f"\n[{index + 1}/{len(selected_df)}] {candidate_id}"
    )

    # If all three arrays already exist, validate and skip.
    if (
        image_path.exists()
        and mask_path.exists()
        and validity_path.exists()
    ):

        try:

            image_check = np.load(image_path)
            mask_check = np.load(mask_path)
            validity_check = np.load(validity_path)

            if (
                image_check.shape == sample_image.shape
                and mask_check.shape == sample_mask.shape
                and validity_check.shape == sample_validity.shape
            ):

                print("Already processed — skipping.")
                success_count += 1
                continue

        except Exception:
            pass

    # Partial outputs are not silently overwritten.
    if (
        image_path.exists()
        or mask_path.exists()
        or validity_path.exists()
    ):

        raise RuntimeError(
            f"{candidate_id}: partial output files exist. "
            "Inspect these three files before rerunning."
        )

    try:

        raw_data, download_status = download_one_tile(row)

        print(
            f"Download status: {download_status}"
        )

        if download_status == "REUSED":
            reused_count += 1

        image, mask, validity = preprocess_tile(
            raw_data,
            candidate_id
        )

        # Confirm labels.
        label_values = np.unique(mask)

        if not np.all(
            np.isin(label_values, [0, 1, 2, 3, 4, 5, 255])
        ):
            raise ValueError(
                f"{candidate_id}: unexpected label values "
                f"{label_values}"
            )

        if np.sum(mask != 255) == 0:
            raise ValueError(
                f"{candidate_id}: no valid WorldCover labels."
            )

        # Save all three arrays.
        np.save(image_path, image)
        np.save(mask_path, mask)
        np.save(validity_path, validity)

        print("Image:", image.shape, image.dtype)
        print("Mask:", mask.shape, mask.dtype)
        print("Validity:", validity.shape, validity.dtype)

        print(
            "Valid label pixels:",
            int(np.sum(mask != 255))
        )

        print(
            "Class IDs:",
            np.unique(mask)
        )

        print("SUCCESS")

        success_count += 1

    except Exception as error:

        print("FAILED:", error)

        failed_records.append({
            "candidate_id": candidate_id,
            "city": row["city"],
            "error": str(error)
        })


# =============================================================================
# 12. SAVE DOWNLOAD/PROCESSING LOG
# =============================================================================

log_path = (
    PROJECT_DIR
    / "exp20_new_real_tiles"
    / "exp20_direct_download_processing_log.csv"
)

pd.DataFrame(failed_records).to_csv(
    log_path,
    index=False
)

print("\nProcessing log:", log_path)


# =============================================================================
# 13. VERIFY FINAL DATASET COUNTS AND FILE MATCHING
# =============================================================================

print("\n" + "=" * 85)
print("FINAL DATASET VERIFICATION")
print("=" * 85)

final_counts = {}

for split in ["train", "val", "test"]:

    image_stems = get_npy_stems(
        FINAL_DATASET / "images" / split
    )

    mask_stems = get_npy_stems(
        FINAL_DATASET / "masks" / split
    )

    validity_stems = get_npy_stems(
        FINAL_DATASET / "validity" / split
    )

    if not (
        image_stems == mask_stems == validity_stems
    ):

        raise RuntimeError(
            f"Final file mismatch in {split} split."
        )

    final_counts[split] = len(image_stems)

    print(
        f"{split.upper():5s}: {len(image_stems)} tiles"
    )


# Expected counts depend on whether all 69 were successfully added.
expected_train = 359 + success_count

if final_counts["train"] != expected_train:
    raise RuntimeError(
        f"Training count mismatch: expected {expected_train}, "
        f"found {final_counts['train']}."
    )

if final_counts["val"] != 80:
    raise RuntimeError(
        f"Validation count changed: {final_counts['val']}"
    )

if final_counts["test"] != 100:
    raise RuntimeError(
        f"Test count changed: {final_counts['test']}"
    )

# Explicitly ensure E20 tiles were not added to validation/test.
e20_ids = {
    safe_filename(x)
    for x in selected_df["candidate_id"].astype(str)
}

val_stems = get_npy_stems(
    FINAL_DATASET / "images" / "val"
)

test_stems = get_npy_stems(
    FINAL_DATASET / "images" / "test"
)

if e20_ids.intersection(val_stems):
    raise RuntimeError(
        "At least one E20 tile was found in validation."
    )

if e20_ids.intersection(test_stems):
    raise RuntimeError(
        "At least one E20 tile was found in test."
    )


# =============================================================================
# 14. CALCULATE ACTUAL CLASS DISTRIBUTION
# =============================================================================

def calculate_class_distribution(mask_folder):

    counts = np.zeros(NUM_CLASSES, dtype=np.int64)

    ignored_pixels = 0

    for mask_path in sorted(mask_folder.glob("*.npy")):

        mask = np.load(mask_path)

        valid = (
            (mask >= 0)
            & (mask < NUM_CLASSES)
        )

        ignored_pixels += int((~valid).sum())

        if valid.any():

            values, frequencies = np.unique(
                mask[valid],
                return_counts=True
            )

            for value, frequency in zip(values, frequencies):

                counts[int(value)] += int(frequency)

    total = counts.sum()

    percentages = (
        counts / total * 100
        if total > 0
        else np.zeros(NUM_CLASSES)
    )

    return counts, percentages, ignored_pixels


distribution_rows = []

for split in ["train", "val", "test"]:

    counts, percentages, ignored = (
        calculate_class_distribution(
            FINAL_DATASET / "masks" / split
        )
    )

    print(f"\n{split.upper()} class distribution:")

    for class_id, class_name in enumerate(CLASS_NAMES):

        print(
            f"{class_name:25s}: "
            f"{counts[class_id]:12,d} pixels | "
            f"{percentages[class_id]:7.3f}%"
        )

        distribution_rows.append({
            "split": split,
            "class_id": class_id,
            "class_name": class_name,
            "pixels": int(counts[class_id]),
            "percentage": float(percentages[class_id])
        })

    print("Ignored/invalid pixels:", f"{ignored:,}")


distribution_path = (
    FINAL_DATASET / "class_distribution_actual.csv"
)

pd.DataFrame(distribution_rows).to_csv(
    distribution_path,
    index=False
)


# =============================================================================
# 15. SAVE DATASET SUMMARY
# =============================================================================

summary = {
    "base_dataset": str(BASE_DATASET),
    "final_dataset": str(FINAL_DATASET),
    "base_train_tiles": 359,
    "base_val_tiles": 80,
    "base_test_tiles": 100,
    "selected_e20_tiles": len(selected_df),
    "successfully_processed_e20_tiles": success_count,
    "reused_downloads": reused_count,
    "failed_tiles": len(failed_records),
    "final_train_tiles": final_counts["train"],
    "final_val_tiles": final_counts["val"],
    "final_test_tiles": final_counts["test"],
    "total_final_tiles": sum(final_counts.values()),
    "image_shape": list(sample_image.shape),
    "mask_shape": list(sample_mask.shape),
    "validity_shape": list(sample_validity.shape)
}

summary_path = FINAL_DATASET / "dataset_summary.json"

with open(summary_path, "w") as f:
    json.dump(summary, f, indent=4)


# =============================================================================
# 16. FINAL REPORT
# =============================================================================

print("\n" + "=" * 85)
print("EXPERIMENT 20 — FINAL SUMMARY")
print("=" * 85)

print("Final dataset:", FINAL_DATASET)

print("\nTile counts:")
print("Train:", final_counts["train"])
print("Val:  ", final_counts["val"])
print("Test: ", final_counts["test"])
print("Total:", sum(final_counts.values()))

print("\nE20 tiles:")
print("Selected:", len(selected_df))
print("Processed/reused:", success_count)
print("Failed:", len(failed_records))

print("\nExpected final dataset if all 69 succeed:")
print("Train: 428")
print("Val:    80")
print("Test:  100")
print("Total: 608")

print("\nSaved files:")
print(" -", distribution_path)
print(" -", summary_path)
print(" -", log_path)

if failed_records:

    print("\nWARNING: Some tiles failed.")

    print(
        "Fix the listed failures and rerun this same cell. "
        "Already processed tiles will be skipped."
    )

else:

    print("\nALL SELECTED E20 TILES PROCESSED SUCCESSFULLY.")

    if final_counts == {
        "train": 428,
        "val": 80,
        "test": 100
    }:

        print(
            "\nFINAL DATASET VERIFIED: 428 TRAIN + "
            "80 VAL + 100 TEST = 608 TILES."
        )

print("\nExperiment 20 pipeline finished.")


EXPERIMENT 20 — DIRECT DOWNLOAD AND DATASET MERGE
Base dataset: /home/jovyan/vault/LULC_Project/processed_dataset_final_expanded
Selected CSV: /home/jovyan/vault/LULC_Project/exp20_new_real_tiles/exp20_selected_tiles.csv
Raw download folder: /home/jovyan/vault/LULC_Project/exp20_new_real_tiles/direct_download
Final dataset: /home/jovyan/vault/LULC_Project/processed_dataset_unified

Selected tiles: 69


FileNotFoundError: Missing dataset folder: /home/jovyan/vault/LULC_Project/processed_dataset_final_expanded/images/train

In [14]:
from pathlib import Path

PROJECT_DIR = Path("/home/jovyan/vault/LULC_Project")

BASE_DATASET = PROJECT_DIR / "processed_dataset_final_expanded"

print("Dataset exists:", BASE_DATASET.exists())
print("\nTop-level contents:")

for item in sorted(BASE_DATASET.iterdir()):
    print(
        f"{'[DIR] ' if item.is_dir() else '[FILE]'} {item.name}"
    )

print("\nRecursive directory structure (first 3 levels):")

for item in sorted(BASE_DATASET.rglob("*")):
    if item.is_dir():
        relative = item.relative_to(BASE_DATASET)
        if len(relative.parts) <= 3:
            print("DIR :", relative)

print("\nNumPy files found:")

for item in sorted(BASE_DATASET.rglob("*.npy"))[:20]:
    print(item.relative_to(BASE_DATASET))

print("\nTotal NumPy files:", len(list(BASE_DATASET.rglob("*.npy"))))

Dataset exists: True

Top-level contents:
[FILE] class_distribution.csv
[FILE] dataset_metadata.json
[FILE] final_preprocessing_class_distribution.csv
[FILE] final_preprocessing_report.json
[FILE] final_preprocessing_tile_audit.csv
[DIR]  test
[FILE] tile_level_audit.csv
[DIR]  train
[FILE] training_class_weights_recommended.json
[DIR]  val

Recursive directory structure (first 3 levels):
DIR : test
DIR : test/images
DIR : test/masks
DIR : test/validity
DIR : train
DIR : train/images
DIR : train/masks
DIR : train/validity
DIR : val
DIR : val/images
DIR : val/masks
DIR : val/validity

NumPy files found:
test/images/addis_ababa_01.npy
test/images/addis_ababa_02.npy
test/images/addis_ababa_03.npy
test/images/addis_ababa_04.npy
test/images/addis_ababa_05.npy
test/images/addis_ababa_06.npy
test/images/addis_ababa_07.npy
test/images/addis_ababa_08.npy
test/images/addis_ababa_09.npy
test/images/addis_ababa_10.npy
test/images/auckland_01.npy
test/images/auckland_02.npy
test/images/auckland_03.

In [15]:
from pathlib import Path
import json
import numpy as np

ROOT = Path("/home/jovyan/vault/LULC_Project/processed_dataset_final_expanded")

# 1. Read preprocessing metadata
for filename in [
    "dataset_metadata.json",
    "final_preprocessing_report.json"
]:
    path = ROOT / filename
    print(f"\n{'='*70}\n{filename}\n{'='*70}")
    if path.exists():
        with open(path, "r") as f:
            print(json.dumps(json.load(f), indent=2))
    else:
        print("Not found")

# 2. Inspect one existing training tile
image_path = sorted((ROOT / "train" / "images").glob("*.npy"))[0]
mask_path = ROOT / "train" / "masks" / image_path.name
validity_path = ROOT / "train" / "validity" / image_path.name

image = np.load(image_path)
mask = np.load(mask_path)
validity = np.load(validity_path)

print("\nSAMPLE TILE:", image_path.name)
print("Image shape:", image.shape, "dtype:", image.dtype)
print("Image min/max:", np.nanmin(image), np.nanmax(image))
print("Image mean/std:", np.nanmean(image), np.nanstd(image))
print("VV min/max:", np.nanmin(image[:, 0]), np.nanmax(image[:, 0]))
print("VH min/max:", np.nanmin(image[:, 1]), np.nanmax(image[:, 1]))
print("Mask shape:", mask.shape, "dtype:", mask.dtype)
print("Mask classes:", np.unique(mask, return_counts=True))
print("Validity shape:", validity.shape, "dtype:", validity.dtype)
print("Validity values:", np.unique(validity, return_counts=True))


dataset_metadata.json
{
  "dataset_name": "Sentinel-1 LULC Expanded Final Dataset",
  "created_at": "2026-11-08T14:13:45.938529",
  "path": "/home/jovyan/vault/LULC_Project/processed_dataset_final_expanded",
  "splits": {
    "train": 359,
    "val": 80,
    "test": 100
  },
  "input_shape": [
    12,
    2,
    256,
    256
  ],
  "mask_shape": [
    256,
    256
  ],
  "validity_shape": [
    12,
    2,
    256,
    256
  ],
  "class_names": {
    "0": "Tree cover",
    "1": "Shrubland / grassland / moss",
    "2": "Cropland",
    "3": "Built-up",
    "4": "Bare land / snow / ice",
    "5": "Water / wetland / mangrove"
  },
  "ignore_label": 255,
  "normalization": {
    "method": "Original training-only median/IQR",
    "statistics": {
      "VV": {
        "median": -10.296875,
        "q25": -13.8203125,
        "q75": -8.0703125,
        "iqr": 5.75
      },
      "VH": {
        "median": -16.9375,
        "q25": -21.4375,
        "q75": -14.671875,
        "iqr": 6.765625
    

In [16]:
# ================================================================
# BUILD UNIFIED LULC DATASET
#
# Base:  processed_dataset_final_expanded
# Add:   69 E20 tiles downloaded directly from Earth Engine
# Output: processed_dataset_unified
#
# Expected output:
# Train: 428
# Val:    80
# Test:  100
#
# Original dataset and checkpoints are never modified.
# ================================================================

import os
import re
import json
import shutil
import zipfile
import requests
import numpy as np
import pandas as pd
import rasterio
import ee

from pathlib import Path
from pyproj import Transformer
from rasterio.windows import Window
from tqdm.auto import tqdm


# ================================================================
# 1. PATHS AND SETTINGS
# ================================================================

PROJECT_DIR = Path("/home/jovyan/vault/LULC_Project")

BASE_DATASET = PROJECT_DIR / "processed_dataset_final_expanded"

SELECTED_CSV = (
    PROJECT_DIR
    / "exp20_new_real_tiles"
    / "exp20_selected_tiles.csv"
)

RAW_DIR = (
    PROJECT_DIR
    / "exp20_new_real_tiles"
    / "direct_download_unified"
)

OUTPUT_DIR = PROJECT_DIR / "processed_dataset_unified"

EE_PROJECT = "final-project-513714"

EXPECTED_NEW_TILES = 69

TILE_SIZE = 256
PIXEL_SIZE = 30
TILE_METERS = TILE_SIZE * PIXEL_SIZE

N_CLASSES = 6
IGNORE_LABEL = 255

CLASS_NAMES = {
    0: "Tree cover",
    1: "Shrubland / grassland / moss",
    2: "Cropland",
    3: "Built-up",
    4: "Bare land / snow / ice",
    5: "Water / wetland / mangrove"
}

# Existing training-only normalization statistics.
VV_MEDIAN = -10.296875
VV_IQR = 5.75

VH_MEDIAN = -16.9375
VH_IQR = 6.765625

# Existing dataset uses:
# (raw_value - training_median) / training_IQR

# IMPORTANT:
# Confirm this arithmetic formula against the original preprocessing
# code before using the unified dataset for model training.


# ================================================================
# 2. VALIDATE INPUTS
# ================================================================

assert BASE_DATASET.exists(), f"Missing base dataset: {BASE_DATASET}"
assert SELECTED_CSV.exists(), f"Missing selected CSV: {SELECTED_CSV}"

assert not OUTPUT_DIR.exists(), (
    f"Output already exists: {OUTPUT_DIR}\n"
    "This script will not overwrite an existing dataset."
)

for split in ["train", "val", "test"]:
    for folder in ["images", "masks", "validity"]:
        p = BASE_DATASET / split / folder
        assert p.exists(), f"Missing folder: {p}"

selected = pd.read_csv(SELECTED_CSV)

assert len(selected) == EXPECTED_NEW_TILES, (
    f"Expected {EXPECTED_NEW_TILES} selected tiles, "
    f"found {len(selected)}"
)

required_columns = [
    "candidate_id",
    "city",
    "latitude",
    "longitude",
    "utm_epsg"
]

for col in required_columns:
    assert col in selected.columns, f"Missing CSV column: {col}"

print("Input validation passed.")
print("Selected new tiles:", len(selected))


# ================================================================
# 3. INITIALIZE EARTH ENGINE
# ================================================================

try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)

print("Earth Engine initialized.")


# ================================================================
# 4. EARTH ENGINE DATA SOURCES
# ================================================================

S1 = ee.ImageCollection("COPERNICUS/S1_GRD")

WORLDCOVER = ee.ImageCollection(
    "ESA/WorldCover/v200"
).first().select("Map")


# WorldCover -> six merged classes
WC_FROM = [10, 20, 30, 40, 50, 60, 70, 80, 90, 95, 100]
WC_TO   = [ 0,  1,  1,  2,  3,  4,  4,  5,  5,  5,   1]


# ================================================================
# 5. BUILD A TILE IMAGE
# ================================================================

def build_tile_image(latitude, longitude, epsg):

    crs = f"EPSG:{epsg}"

    transformer = Transformer.from_crs(
        "EPSG:4326",
        crs,
        always_xy=True
    )

    center_x, center_y = transformer.transform(
        longitude,
        latitude
    )

    xmin = center_x - TILE_METERS / 2
    xmax = center_x + TILE_METERS / 2

    ymin = center_y - TILE_METERS / 2
    ymax = center_y + TILE_METERS / 2

    transform = [
        PIXEL_SIZE, 0, xmin,
        0, -PIXEL_SIZE, ymax
    ]

    region = ee.Geometry.Rectangle(
        [xmin, ymin, xmax, ymax],
        proj=crs,
        geodesic=False
    )

    sar_bands = []
    validity_bands = []

    # ------------------------------------------------------------
    # Twelve monthly VV/VH composites for 2021
    # ------------------------------------------------------------

    for month in range(1, 13):

        start = ee.Date.fromYMD(2021, month, 1)
        end = start.advance(1, "month")

        collection = (
            S1
            .filterBounds(region)
            .filterDate(start, end)
            .filter(ee.Filter.eq("instrumentMode", "IW"))
            .filter(
                ee.Filter.listContains(
                    "transmitterReceiverPolarisation",
                    "VV"
                )
            )
            .filter(
                ee.Filter.listContains(
                    "transmitterReceiverPolarisation",
                    "VH"
                )
            )
            .select(["VV", "VH"])
        )

        # An empty collection becomes a fully masked image.
        # Its validity bands will therefore be zero.
        empty_image = (
            ee.Image.constant([-9999, -9999])
            .rename(["VV", "VH"])
            .updateMask(ee.Image.constant(0))
        )

        monthly = ee.Image(
            ee.Algorithms.If(
                collection.size().gt(0),
                collection.median(),
                empty_image
            )
        ).select(["VV", "VH"])

        vv = monthly.select("VV")
        vh = monthly.select("VH")

        vv_valid = (
            vv.mask()
            .unmask(0)
            .rename(f"VV_valid_{month:02d}")
            .toFloat()
        )

        vh_valid = (
            vh.mask()
            .unmask(0)
            .rename(f"VH_valid_{month:02d}")
            .toFloat()
        )

        vv_data = (
            vv.unmask(-9999)
            .rename(f"VV_{month:02d}")
            .toFloat()
        )

        vh_data = (
            vh.unmask(-9999)
            .rename(f"VH_{month:02d}")
            .toFloat()
        )

        sar_bands.extend([vv_data, vh_data])
        validity_bands.extend([vv_valid, vh_valid])

    # ------------------------------------------------------------
    # ESA WorldCover labels
    # ------------------------------------------------------------

    labels = (
        WORLDCOVER
        .remap(
            WC_FROM,
            WC_TO,
            IGNORE_LABEL
        )
        .rename("label")
        .toUint8()
    )

    # 24 SAR + 24 validity + 1 label = 49 bands
    combined = ee.Image.cat(
        sar_bands + validity_bands + [labels]
    )

    return combined, crs, transform, region


# ================================================================
# 6. DOWNLOAD HELPERS
# ================================================================

def safe_filename(value):
    return re.sub(r"[^A-Za-z0-9_-]+", "_", str(value))


def download_geotiff(image, tile_name):

    url = image.getDownloadURL({
        "name": tile_name,
        "crs": CURRENT_CRS,
        "crs_transform": CURRENT_TRANSFORM,
        "region": CURRENT_REGION,
        "format": "GEO_TIFF"
    })

    response = requests.get(
        url,
        timeout=300
    )

    response.raise_for_status()

    tile_dir = RAW_DIR / tile_name
    tile_dir.mkdir(parents=True, exist_ok=True)

    content = response.content

    # Earth Engine may return a ZIP archive.
    if content[:2] == b"PK":

        zip_path = tile_dir / "download.zip"

        with open(zip_path, "wb") as f:
            f.write(content)

        with zipfile.ZipFile(zip_path) as z:
            tif_files = [
                n for n in z.namelist()
                if n.lower().endswith((".tif", ".tiff"))
            ]

            assert tif_files, "No TIFF found in downloaded ZIP."

            tif_name = tif_files[0]

            with z.open(tif_name) as src:
                output_path = tile_dir / "tile.tif"

                with open(output_path, "wb") as dst:
                    shutil.copyfileobj(src, dst)

        return output_path

    output_path = tile_dir / "tile.tif"

    with open(output_path, "wb") as f:
        f.write(content)

    return output_path


# ================================================================
# 7. CONVERT GEOTIFF TO MODEL ARRAYS
# ================================================================

def convert_tile(tif_path, expected_crs, expected_transform):

    with rasterio.open(tif_path) as src:

        if src.count != 49:
            raise ValueError(
                f"Expected 49 bands, received {src.count}"
            )

        if src.crs is None:
            raise ValueError("GeoTIFF has no CRS.")

        if src.crs.to_string() != expected_crs:
            raise ValueError(
                f"CRS mismatch: {src.crs} vs {expected_crs}"
            )

        if src.width not in (256, 257):
            raise ValueError(f"Unexpected width: {src.width}")

        if src.height not in (256, 257):
            raise ValueError(f"Unexpected height: {src.height}")

        # Verify exact pixel size and top-left origin.
        actual = src.transform

        expected = rasterio.Affine(
            expected_transform[0],
            expected_transform[1],
            expected_transform[2],
            expected_transform[3],
            expected_transform[4],
            expected_transform[5]
        )

        tolerance = 0.01

        for a, b in zip(actual[:6], expected[:6]):
            if abs(a - b) > tolerance:
                raise ValueError(
                    "GeoTIFF transform is not aligned with the "
                    "requested 30 m grid.\n"
                    f"Actual:   {actual}\n"
                    f"Expected: {expected}"
                )

        # Crop only the extra bottom/right edge pixel.
        # No resizing or interpolation.
        data = src.read(
            window=Window(0, 0, 256, 256)
        )

    if data.shape != (49, 256, 256):
        raise ValueError(f"Unexpected array shape: {data.shape}")

    # ------------------------------------------------------------
    # First 24 bands: SAR
    # Next 24 bands: validity
    # Last band: labels
    # ------------------------------------------------------------

    sar = data[:24].astype(np.float32)
    validity = data[24:48]
    labels = data[48]

    sar = sar.reshape(12, 2, 256, 256)
    validity = validity.reshape(12, 2, 256, 256)

    validity = (
        np.isfinite(validity)
        & (validity > 0.5)
    ).astype(np.uint8)

    # Convert raw dB values to the existing normalized scale.
    vv = sar[:, 0]
    vh = sar[:, 1]

    vv_valid = validity[:, 0] == 1
    vh_valid = validity[:, 1] == 1

    vv_normalized = (
        vv - VV_MEDIAN
    ) / VV_IQR

    vh_normalized = (
        vh - VH_MEDIAN
    ) / VH_IQR

    # Invalid observations are stored as zero, with validity
    # separately indicating that they are unavailable.
    vv_normalized[~vv_valid] = 0.0
    vh_normalized[~vh_valid] = 0.0

    image = np.stack(
        [vv_normalized, vh_normalized],
        axis=1
    ).astype(np.float32)

    # ------------------------------------------------------------
    # Labels
    # ------------------------------------------------------------

    labels = labels.astype(np.uint8)

    valid_classes = np.isin(
        labels,
        np.arange(N_CLASSES)
    )

    labels[~valid_classes] = IGNORE_LABEL

    return image, labels, validity


# ================================================================
# 8. COPY THE BASE DATASET WITHOUT MODIFYING IT
# ================================================================

print("\nCopying expanded dataset...")

shutil.copytree(
    BASE_DATASET,
    OUTPUT_DIR
)

print("Base dataset copied.")


# ================================================================
# 9. DOWNLOAD AND ADD THE 69 NEW TRAINING TILES
# ================================================================

RAW_DIR.mkdir(parents=True, exist_ok=True)

success = []
failed = []

for _, row in tqdm(
    selected.iterrows(),
    total=len(selected),
    desc="Downloading E20 tiles"
):

    tile_id = safe_filename(row["candidate_id"])

    try:

        # Skip a tile only if all three output arrays exist.
        output_image = (
            OUTPUT_DIR / "train" / "images" / f"{tile_id}.npy"
        )

        output_mask = (
            OUTPUT_DIR / "train" / "masks" / f"{tile_id}.npy"
        )

        output_validity = (
            OUTPUT_DIR / "train" / "validity" / f"{tile_id}.npy"
        )

        if (
            output_image.exists()
            and output_mask.exists()
            and output_validity.exists()
        ):
            success.append(tile_id)
            continue

        epsg_value = str(row["utm_epsg"]).replace("EPSG:", "")
        epsg = int(float(epsg_value))

        combined, crs, transform, region = build_tile_image(
            float(row["latitude"]),
            float(row["longitude"]),
            epsg
        )

        # Set the per-tile values used by the download helper.
        CURRENT_CRS = crs
        CURRENT_TRANSFORM = transform
        CURRENT_REGION = region

        tif_path = download_geotiff(
            combined,
            tile_id
        )

        image, mask, validity = convert_tile(
            tif_path,
            crs,
            transform
        )

        # Final shape checks.
        assert image.shape == (12, 2, 256, 256)
        assert mask.shape == (256, 256)
        assert validity.shape == (12, 2, 256, 256)

        assert image.dtype == np.float32
        assert mask.dtype == np.uint8
        assert validity.dtype == np.uint8

        # Save to training only.
        np.save(output_image, image)
        np.save(output_mask, mask)
        np.save(output_validity, validity)

        success.append(tile_id)

    except Exception as exc:

        failed.append({
            "tile": tile_id,
            "error": str(exc)
        })

        print(f"\nFAILED: {tile_id}")
        print(exc)


# ================================================================
# 10. VERIFY THE UNIFIED DATASET
# ================================================================

print("\n" + "=" * 70)
print("UNIFIED DATASET VERIFICATION")
print("=" * 70)

expected_counts = {
    "train": 428,
    "val": 80,
    "test": 100
}

actual_counts = {}

for split in ["train", "val", "test"]:

    image_dir = OUTPUT_DIR / split / "images"
    mask_dir = OUTPUT_DIR / split / "masks"
    validity_dir = OUTPUT_DIR / split / "validity"

    images = {
        p.stem for p in image_dir.glob("*.npy")
    }

    masks = {
        p.stem for p in mask_dir.glob("*.npy")
    }

    validities = {
        p.stem for p in validity_dir.glob("*.npy")
    }

    assert images == masks, f"{split}: image/mask mismatch"
    assert images == validities, f"{split}: image/validity mismatch"

    actual_counts[split] = len(images)

    print(
        f"{split.upper():5s}: "
        f"{len(images)} images, "
        f"{len(masks)} masks, "
        f"{len(validities)} validity arrays"
    )

    assert len(images) == expected_counts[split], (
        f"{split} expected {expected_counts[split]}, "
        f"found {len(images)}"
    )

assert len(success) + len([
    x for x in selected["candidate_id"]
    if safe_filename(x) in success
]) >= 0  # Summary-only consistency check.


# ================================================================
# 11. COMPUTE ACTUAL CLASS DISTRIBUTION
# ================================================================

class_pixels = {
    split: np.zeros(N_CLASSES, dtype=np.int64)
    for split in ["train", "val", "test"]
}

ignored_pixels = {
    split: 0
    for split in ["train", "val", "test"]
}

for split in ["train", "val", "test"]:

    mask_dir = OUTPUT_DIR / split / "masks"

    for mask_path in tqdm(
        sorted(mask_dir.glob("*.npy")),
        desc=f"Counting {split}"
    ):

        mask = np.load(mask_path)

        for class_id in range(N_CLASSES):
            class_pixels[split][class_id] += np.sum(
                mask == class_id
            )

        ignored_pixels[split] += np.sum(
            mask == IGNORE_LABEL
        )

distribution = []

for split in ["train", "val", "test"]:

    total_valid = class_pixels[split].sum()

    for class_id in range(N_CLASSES):

        count = int(class_pixels[split][class_id])

        percentage = (
            100 * count / total_valid
            if total_valid > 0 else 0
        )

        distribution.append({
            "split": split,
            "class_id": class_id,
            "class_name": CLASS_NAMES[class_id],
            "pixels": count,
            "percentage": percentage
        })

distribution_df = pd.DataFrame(distribution)

distribution_df.to_csv(
    OUTPUT_DIR / "unified_class_distribution.csv",
    index=False
)

print("\nClass distribution:")
print(
    distribution_df.pivot(
        index="class_name",
        columns="split",
        values="percentage"
    ).round(3)
)


# ================================================================
# 12. SAVE DATASET METADATA
# ================================================================

metadata = {
    "dataset_name": "Sentinel-1 LULC Unified Dataset",
    "base_dataset": str(BASE_DATASET),
    "output_dataset": str(OUTPUT_DIR),
    "new_tiles_expected": EXPECTED_NEW_TILES,
    "new_tiles_downloaded": len(success),
    "failed_tiles": failed,
    "splits": actual_counts,
    "input_shape": [12, 2, 256, 256],
    "mask_shape": [256, 256],
    "validity_shape": [12, 2, 256, 256],
    "class_names": CLASS_NAMES,
    "ignore_label": IGNORE_LABEL,
    "normalization": {
        "method": "Training-only median/IQR",
        "formula": "(value - median) / IQR",
        "VV_median": VV_MEDIAN,
        "VV_IQR": VV_IQR,
        "VH_median": VH_MEDIAN,
        "VH_IQR": VH_IQR
    },
    "original_dataset_modified": False,
    "validation_and_test_unchanged": True,
    "test_set_used_for_training": False
}

with open(
    OUTPUT_DIR / "unified_dataset_metadata.json",
    "w"
) as f:
    json.dump(metadata, f, indent=4)


# ================================================================
# 13. FINAL STATUS
# ================================================================

print("\n" + "=" * 70)
print("FINAL STATUS")
print("=" * 70)

print("New tiles successfully saved:", len(success))
print("Failed tiles:", len(failed))
print("Dataset:", OUTPUT_DIR)

if failed:
    print("\nFailed tile details:")
    for item in failed:
        print(item)

if (
    actual_counts["train"] == 428
    and actual_counts["val"] == 80
    and actual_counts["test"] == 100
    and len(failed) == 0
):
    print("\nSUCCESS: Unified dataset contains all 69 new tiles.")
else:
    print("\nINCOMPLETE: Check failed downloads and tile counts.")

Input validation passed.
Selected new tiles: 69
Earth Engine initialized.

Copying expanded dataset...
Base dataset copied.



FAILED: E20_Cairo_023
GeoTIFF transform is not aligned with the requested 30 m grid.
Actual:   | 30.00, 0.00, 365574.88|
| 0.00,-30.00, 3323843.00|
| 0.00, 0.00, 1.00|
Expected: | 30.00, 0.00, 365604.88|
| 0.00,-30.00, 3323843.00|
| 0.00, 0.00, 1.00|

FAILED: E20_Delhi_018
GeoTIFF transform is not aligned with the requested 30 m grid.
Actual:   | 30.00, 0.00, 748753.52|
| 0.00,-30.00, 3186595.02|
| 0.00, 0.00, 1.00|
Expected: | 30.00, 0.00, 748783.52|
| 0.00,-30.00, 3186595.02|
| 0.00, 0.00, 1.00|

FAILED: E20_Delhi_019
GeoTIFF transform is not aligned with the requested 30 m grid.
Actual:   | 30.00, 0.00, 737566.53|
| 0.00,-30.00, 3141441.50|
| 0.00, 0.00, 1.00|
Expected: | 30.00, 0.00, 737596.53|
| 0.00,-30.00, 3141441.50|
| 0.00, 0.00, 1.00|

FAILED: E20_Lisbon_013
GeoTIFF transform is not aligned with the requested 30 m grid.
Actual:   | 30.00, 0.00, 506188.78|
| 0.00,-30.00, 4314405.27|
| 0.00, 0.00, 1.00|
Expected: | 30.00, 0.00, 506218.78|
| 0.00,-30.00, 4314405.27|
| 0.00, 0.0

AssertionError: train expected 428, found 422

In [17]:
from pathlib import Path
import numpy as np
import pandas as pd
import json

PROJECT_DIR = Path("/home/jovyan/vault/LULC_Project")

BASE = PROJECT_DIR / "processed_dataset_final_expanded"
FINAL = PROJECT_DIR / "processed_dataset_unified"

EXPECTED_COUNTS = {
    "train": 422,
    "val": 80,
    "test": 100
}

EXPECTED_SHAPES = {
    "images": (12, 2, 256, 256),
    "masks": (256, 256),
    "validity": (12, 2, 256, 256)
}

EXPECTED_DTYPES = {
    "images": np.float32,
    "masks": np.uint8,
    "validity": np.uint8
}

results = []
all_checks_passed = True


def get_ids(folder):
    return {
        p.stem for p in folder.glob("*.npy")
    }


def compare_arrays(path_a, path_b):
    a = np.load(path_a, mmap_mode="r")
    b = np.load(path_b, mmap_mode="r")

    return (
        a.shape == b.shape
        and a.dtype == b.dtype
        and np.array_equal(a, b)
    )


print("=" * 75)
print("UNIFIED DATASET — COMPLETE VERIFICATION")
print("=" * 75)

print("\nFINAL DATASET:", FINAL)
print("BASE DATASET: ", BASE)

assert FINAL.exists(), "Unified dataset folder does not exist."


# ============================================================
# 1. VERIFY ALL SPLITS AND FILE COUNTS
# ============================================================

for split in ["train", "val", "test"]:

    print(f"\n{'-' * 60}")
    print(f"VERIFYING {split.upper()}")
    print("-" * 60)

    final_dirs = {
        kind: FINAL / split / kind
        for kind in ["images", "masks", "validity"]
    }

    base_dirs = {
        kind: BASE / split / kind
        for kind in ["images", "masks", "validity"]
    }

    final_ids = {
        kind: get_ids(folder)
        for kind, folder in final_dirs.items()
    }

    base_ids = {
        kind: get_ids(folder)
        for kind, folder in base_dirs.items()
    }

    # Check matching files within each split.
    matching_files = (
        final_ids["images"]
        == final_ids["masks"]
        == final_ids["validity"]
    )

    print("Images:", len(final_ids["images"]))
    print("Masks:", len(final_ids["masks"]))
    print("Validity:", len(final_ids["validity"]))
    print("Matching filenames:", matching_files)

    if not matching_files:
        print("Image-only:", sorted(
            final_ids["images"] - final_ids["masks"]
        )[:10])
        print("Mask-only:", sorted(
            final_ids["masks"] - final_ids["images"]
        )[:10])
        all_checks_passed = False

    count_ok = (
        len(final_ids["images"]) == EXPECTED_COUNTS[split]
    )

    print(
        f"Expected count: {EXPECTED_COUNTS[split]}",
        "| Count correct:", count_ok
    )

    if not count_ok:
        all_checks_passed = False

    # Verify that every original base tile is still present.
    original_ids = base_ids["images"]

    original_tiles_present = (
        original_ids.issubset(final_ids["images"])
    )

    print("Original base tiles preserved:", original_tiles_present)

    if not original_tiles_present:
        missing = sorted(
            original_ids - final_ids["images"]
        )
        print("Missing original tiles:", missing[:20])
        all_checks_passed = False

    # For validation and test, require exact same tile IDs.
    if split in ["val", "test"]:

        same_ids = (
            final_ids["images"] == original_ids
        )

        print("Same tile IDs as original:", same_ids)

        if not same_ids:
            all_checks_passed = False

        # Check exact array equality for every original tile.
        arrays_identical = True

        for tile_id in sorted(original_ids):

            for kind in ["images", "masks", "validity"]:

                path_a = base_dirs[kind] / f"{tile_id}.npy"
                path_b = final_dirs[kind] / f"{tile_id}.npy"

                if not compare_arrays(path_a, path_b):
                    arrays_identical = False
                    print(
                        "DIFFERENCE:",
                        split, kind, tile_id
                    )
                    break

            if not arrays_identical:
                break

        print(
            "Original arrays unchanged:",
            arrays_identical
        )

        if not arrays_identical:
            all_checks_passed = False

    # For training, identify added tiles.
    if split == "train":

        added_ids = (
            final_ids["images"] - original_ids
        )

        print("New training tiles added:", len(added_ids))

        print("New tile IDs:")
        for tile_id in sorted(added_ids):
            print(" ", tile_id)

        if len(added_ids) != 63:
            all_checks_passed = False

    # Validate every final array's shape and dtype.
    shape_dtype_ok = True

    for kind, folder in final_dirs.items():

        expected_shape = EXPECTED_SHAPES[kind]
        expected_dtype = EXPECTED_DTYPES[kind]

        for path in sorted(folder.glob("*.npy")):

            arr = np.load(path, mmap_mode="r")

            if (
                arr.shape != expected_shape
                or arr.dtype != expected_dtype
            ):
                print(
                    "INVALID ARRAY:",
                    split,
                    kind,
                    path.name,
                    "shape:", arr.shape,
                    "dtype:", arr.dtype
                )

                shape_dtype_ok = False
                break

        if not shape_dtype_ok:
            break

    print("All array shapes and dtypes correct:", shape_dtype_ok)

    if not shape_dtype_ok:
        all_checks_passed = False

    results.append({
        "split": split,
        "images": len(final_ids["images"]),
        "masks": len(final_ids["masks"]),
        "validity": len(final_ids["validity"]),
        "count_correct": count_ok,
        "filenames_match": matching_files,
        "shapes_dtypes_correct": shape_dtype_ok
    })


# ============================================================
# 2. VERIFY TOTAL DATASET COUNT
# ============================================================

total_tiles = sum(
    EXPECTED_COUNTS.values()
)

actual_total = sum(
    x["images"] for x in results
)

print("\n" + "=" * 75)
print("FINAL SUMMARY")
print("=" * 75)

print("Training:   422")
print("Validation:  80")
print("Test:       100")
print("-" * 35)
print("Expected total:", total_tiles)
print("Actual total:  ", actual_total)

print("\nVerification table:")
print(pd.DataFrame(results).to_string(index=False))


# ============================================================
# 3. VERIFY DATASET METADATA
# ============================================================

metadata_path = FINAL / "dataset_metadata.json"

if metadata_path.exists():

    with open(metadata_path) as f:
        metadata = json.load(f)

    print("\nCopied base metadata:")
    print("Dataset name:", metadata.get("dataset_name"))
    print("Original split counts:", metadata.get("splits"))

    print(
        "\nNote: this is copied base metadata, "
        "not necessarily updated unified metadata."
    )

else:
    print("\nNo copied dataset_metadata.json found.")


# ============================================================
# 4. FINAL VERDICT
# ============================================================

print("\n" + "=" * 75)

if all_checks_passed and actual_total == total_tiles:

    print("SUCCESS: ALL DATASET VERIFICATION CHECKS PASSED.")
    print("The unified dataset contains 602 verified tiles.")
    print("Original validation and test arrays are unchanged.")

else:

    print("VERIFICATION INCOMPLETE.")
    print("Review the failed checks above.")

print("=" * 75)

UNIFIED DATASET — COMPLETE VERIFICATION

FINAL DATASET: /home/jovyan/vault/LULC_Project/processed_dataset_unified
BASE DATASET:  /home/jovyan/vault/LULC_Project/processed_dataset_final_expanded

------------------------------------------------------------
VERIFYING TRAIN
------------------------------------------------------------
Images: 422
Masks: 422
Validity: 422
Matching filenames: True
Expected count: 422 | Count correct: True
Original base tiles preserved: True
New training tiles added: 63
New tile IDs:
  E20_Bangkok_001
  E20_Bangkok_017
  E20_Beijing_007
  E20_Bengaluru_003
  E20_Bengaluru_008
  E20_Bengaluru_013
  E20_Bengaluru_018
  E20_Bengaluru_022
  E20_Cairo_002
  E20_Cairo_013
  E20_Cairo_014
  E20_Cairo_021
  E20_Cape_Town_010
  E20_Cape_Town_018
  E20_Cape_Town_020
  E20_Delhi_005
  E20_Delhi_013
  E20_Delhi_014
  E20_Istanbul_006
  E20_Istanbul_010
  E20_Jakarta_011
  E20_Jakarta_012
  E20_Jakarta_015
  E20_Jakarta_017
  E20_Jakarta_019
  E20_Johannesburg_001
  E20_J